# 02 OFFICIAL SPLIT

Extracted from the original notebooks. **Outputs preserved.** Originals are unmodified.


## A · Segmentation on the official TCIA split


**`SEG` cell 0** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 2-OFFICIAL — DS-Attn-UNet + ASPP on the OFFICIAL TCIA SPLIT
#   Architecture / loss / augmentation / optimiser: IDENTICAL to your CV
#   segmentation cell. Only the fold roles change (role_f -> role_of).
#   Trains only on official-training regions; the 378 official test
#   regions are never seen during training or validation.
#   Output -> predmasks_mass_official/     (predmasks_mass/ untouched)
#   Resumable: a finished fold reloads its checkpoint instead of retraining.
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
DS_WEIGHTS=[1.0,0.5,0.3,0.2]
torch.backends.cudnn.benchmark=True

LES="mass"
CSV=os.path.join(D,"unified_folds_%s.csv"%LES)
OUT=os.path.join(D,"predmasks_%s_official"%LES); os.makedirs(OUT,exist_ok=True)
CKD=os.path.join(D,"ckpt_official"); os.makedirs(CKD,exist_ok=True)
ck=lambda k: os.path.join(CKD,"seg_dsaspp_%s_official_fold%d.pth"%(LES,k))

# ═══════════════ VERIFICATION — runs before any GPU work ═══════════════
d=pd.read_csv(CSV).reset_index(drop=True)
print("="*74); print("VERIFICATION — OFFICIAL TCIA SPLIT"); print("="*74)
print("  csv                : %s"%CSV)
assert "official_split" in d.columns, "official_split column missing"
assert "msk" in d.columns and "img" in d.columns, "img/msk columns missing"
_te=d["official_split"].astype(str).str.lower().str.contains("test")
print("  regions            : %d  (official train %d | official test %d)"
      %(len(d),(~_te).sum(),_te.sum()))
print("  patients           : train %d | test %d"
      %(d.patient_id[~_te].nunique(), d.patient_id[_te].nunique()))
ov=len(set(d.patient_id[~_te]) & set(d.patient_id[_te]))
print("  patients in BOTH   : %d"%ov); assert ov==0, "patient overlap in the official split"
assert (~_te).sum()==1318 and _te.sum()==378, \
       "expected 1318/378 official regions, got %d/%d"%((~_te).sum(),_te.sum())
for k in range(5):
    c="role_of%d"%k
    assert c in d.columns, "%s missing - run CELL A first"%c
    r=d[c]
    assert ((r=="test")==_te).all(), "%s test set != official test"%c
    assert not (r.isin(["train","val"]) & _te).any(), "OFFICIAL TEST REGION IN TRAINING (%s)"%c
    assert len(set(d.patient_id[r=="train"]) & set(d.patient_id[r=="test"]))==0, "%s train/test patient leak"%c
    assert len(set(d.patient_id[r=="val"])   & set(d.patient_id[r=="test"]))==0, "%s val/test patient leak"%c
    assert len(set(d.patient_id[r=="train"]) & set(d.patient_id[r=="val"]))==0,  "%s train/val patient leak"%c
    print("  %-9s : train %4d | val %4d | test %3d   (test == official test)"
          %(c,(r=="train").sum(),(r=="val").sum(),(r=="test").sum()))
nm=d["msk"].astype(str).apply(os.path.exists).sum(); ni=d["img"].astype(str).apply(os.path.exists).sum()
print("  files on disk      : img %d/%d | msk %d/%d"%(ni,len(d),nm,len(d)))
assert nm==len(d) and ni==len(d), "some img/msk files are missing"
print("  VERIFIED: training uses official-train regions only.")
print("="*74)

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d,aug,mult=1): s.df=d.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))
class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2)
        s.bn=ASPP(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out =nn.Conv2d(b,2,1)
        s.ds2=nn.Conv2d(b*2,2,1); s.ds3=nn.Conv2d(b*4,2,1); s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        main=s.out(d1)
        if s.training: return main, s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return main

def tversky_ce(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())
def ds_loss(outs,t):
    main,o2,o3,o4=outs
    L=DS_WEIGHTS[0]*tversky_ce(main,t)
    for w,o in zip(DS_WEIGHTS[1:],[o2,o3,o4]):
        td=F.interpolate(t.unsqueeze(1).float(),size=o.shape[2:],mode="nearest").squeeze(1).long()
        L=L+w*tversky_ce(o,td)
    return L
@torch.no_grad()
def sc(net,dd):
    net.eval(); ld=DataLoader(DS(dd,False),batch_size=BATCH,shuffle=False,num_workers=0); r=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1),iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(r)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),
                prec=R.prec.mean(),rec=R.rec.mean())
@torch.no_grad()
def probs(net,dd):
    net.eval(); ld=DataLoader(DS(dd,False),batch_size=BATCH,shuffle=False,num_workers=0)
    out=np.zeros((len(dd),IMG,IMG),np.float32); pos=0
    for x,_,_ in ld:
        x=x.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        p=torch.softmax(o.float(),1)[:,1].cpu().numpy()
        out[pos:pos+len(p)]=p; pos+=len(p)
    return out
def save_mask(row, prob):
    m=(prob>THR).astype(np.uint8)
    cv2.imwrite(os.path.join(OUT, os.path.basename(row["img"]).replace("_img.png","")+"_pred.png"),
                cv2.resize(m*255,(512,512),interpolation=cv2.INTER_NEAREST))
    gt=cv2.imread(row["msk"],cv2.IMREAD_GRAYSCALE)
    gt=(cv2.resize(gt,(IMG,IMG),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
    tp=(m*gt).sum(); fp=(m*(1-gt)).sum(); fn=((1-m)*gt).sum()
    return (2*tp+1)/(2*tp+fp+fn+1)

test_i=np.where(_te.values)[0]
test_sum=np.zeros((len(test_i),IMG,IMG),np.float32)
oof=np.full(len(d),np.nan); fold_dice=[]

for k in range(5):
    role=d["role_of%d"%k]
    tr=d[role=="train"]; va=d[role=="val"]; te=d[role=="test"]
    net=DSAttnUNet().to(DEV)
    if os.path.exists(ck(k)):
        net.load_state_dict({q:v.to(DEV) for q,v in torch.load(ck(k),map_location="cpu").items()})
        print("\n### fold %d — checkpoint found, skipping training"%k)
    else:
        print("\n### fold %d | train %d x%d | val %d | test %d"%(k,len(tr),MULT,len(va),len(te)))
        t0=time.time(); torch.manual_seed(k); np.random.seed(k)
        tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
        scaler=torch.amp.GradScaler(); opt=torch.optim.Adam(net.parameters(),lr=LR)
        sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
        best,bs,ni=0.,None,0
        for ep in range(1,EPOCHS+1):
            net.train(); tot=0.; nb=0
            for x,y,_ in tl:
                x=x.to(DEV); y=y.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"): outs=net(x); l=ds_loss(outs,y)
                scaler.scale(l).backward(); scaler.step(opt); scaler.update()
                tot+=l.item(); nb+=1
            vd=sc(net,va)["dice"]; sch.step(vd)
            if vd>best: best=vd; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            print("  ep %2d | loss %.4f | val-Dice %.4f%s"%(ep,tot/max(nb,1),vd," *" if vd==best else ""))
            if ni>=10: print("  early stop"); break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        torch.save({q:v.cpu() for q,v in net.state_dict().items()}, ck(k))
        print("  fold %d trained in %.0fs — checkpoint saved"%(k,time.time()-t0))

    m=sc(net,te); fold_dice.append(m["dice"])
    print("  official TEST Dice %.4f (median %.4f) | IoU %.4f | P %.3f | R %.3f"
          %(m["dice"],m["median"],m["iou"],m["prec"],m["rec"]))
    pv=probs(net,va)
    for i,gi in enumerate(va.index.values):
        oof[gi]=save_mask(d.iloc[gi], pv[i])
    test_sum += probs(net,te)
    del net; torch.cuda.empty_cache()

print("\nwriting 5-fold averaged masks for the %d official TEST regions..."%len(test_i))
for i,gi in enumerate(test_i):
    oof[gi]=save_mask(d.iloc[gi], test_sum[i]/5.0)

assert not np.isnan(oof).any(), "%d regions got no mask"%int(np.isnan(oof).sum())
d["oof_dice_official"]=oof
d.to_csv(CSV,index=False)

tr_m=oof[~_te.values].mean(); te_m=oof[_te.values].mean()
print("\n"+"="*74)
print("SEGMENTATION ON THE OFFICIAL TCIA SPLIT — DS-Attn-UNet + ASPP")
print("="*74)
print("  per-fold official TEST Dice : %s"%[round(x,4) for x in fold_dice])
print("  MEAN over folds             : %.4f +/- %.4f"%(np.mean(fold_dice),np.std(fold_dice)))
print("  5-fold averaged masks, TEST : %.4f   (n=%d)  <- report this"%(te_m,_te.sum()))
print("  out-of-fold masks,   TRAIN  : %.4f   (n=%d)"%(tr_m,(~_te).sum()))
print("  CV reference                : 0.900")
print("  masks -> %s"%OUT)
print("="*74)

VERIFICATION — OFFICIAL TCIA SPLIT
  csv                : /root/autodl-tmp/CBIS/unified_folds_mass.csv
  regions            : 1696  (official train 1318 | official test 378)
  patients           : train 691 | test 201
  patients in BOTH   : 0
  role_of0  : train 1054 | val  264 | test 378   (test == official test)
  role_of1  : train 1054 | val  264 | test 378   (test == official test)
  role_of2  : train 1053 | val  265 | test 378   (test == official test)
  role_of3  : train 1057 | val  261 | test 378   (test == official test)
  role_of4  : train 1054 | val  264 | test 378   (test == official test)
  files on disk      : img 1696/1696 | msk 1696/1696
  VERIFIED: training uses official-train regions only.

### fold 0 | train 1054 x8 | val 264 | test 378
  ep  1 | loss 0.3624 | val-Dice 0.8602 *
  ep  2 | loss 0.2944 | val-Dice 0.8461
  ep  3 | loss 0.2759 | val-Dice 0.8724 *
  ep  4 | loss 0.2654 | val-Dice 0.8743 *
  ep  5 | loss 0.2550 | val-Dice 0.8735
  ep  6 | loss 0.2435 | val-D

## B · Wide-context crops from the official masks


**`SEG` cell 8** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 20A-OFFICIAL — WIDE-CONTEXT MASS CROPS from the OFFICIAL-SPLIT MASKS
#   Identical geometry to your original cell 20A. The only changes:
#     PM  -> predmasks_mass_official      (masks from the official-split segmenter)
#     OUT -> crops_wide_mass_official     (your crops_wide_mass/ is untouched)
#   The projected mask is PREDICTED, never ground truth.
#   CPU only. ~20-30 min.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
cv2.setNumThreads(0)

D    = "/root/autodl-tmp/CBIS"
JP   = os.path.join(D, "jpeg")
LES  = "mass"
PM   = os.path.join(D, "predmasks_%s_official" % LES)              # <-- OFFICIAL masks
OUT  = os.path.join(D, "crops_wide_%s_official" % LES); os.makedirs(OUT, exist_ok=True)
FIG  = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)

S          = 512
WIDE_MULT  = 1.75

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
for c in ["img", "msk", "full_series", "mask_series"]:
    assert c in d.columns, "column '%s' missing from unified_folds_%s.csv" % (c, LES)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))

# ═══════════════════ VERIFICATION ═══════════════════
print("="*66); print("VERIFICATION"); print("="*66)
assert os.path.isdir(PM), "official mask folder not found: %s\nRun the segmentation cell first." % PM
npm = int(d["predmask"].apply(os.path.exists).sum())
print("  mask source        : %s" % PM)
print("  official masks     : %d / %d present" % (npm, len(d)))
assert npm == len(d), "%d official masks missing — the segmentation cell did not finish" % (len(d)-npm)
_te = d["official_split"].astype(str).str.lower().str.contains("test")
print("  official split     : train %d | test %d regions" % ((~_te).sum(), _te.sum()))
print("  patients in BOTH   : %d" % len(set(d.patient_id[~_te]) & set(d.patient_id[_te])))
print("  output folder      : %s" % OUT)
print("  (this cell only re-crops; the masks it projects are PREDICTED, never ground truth)")
print("="*66 + "\n")


def series_files(uid):
    p = os.path.join(JP, str(uid))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []


def read_full(uid):
    best, ba = None, -1
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > ba:
            best, ba = im, im.size
    return best


def read_maskseries(uid, ref_shape):
    """mask series can contain BOTH a crop and the mask - pick by shape + binariness"""
    c = []
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None:
            continue
        score = 2.0 * float(im.shape == ref_shape) + float(((im < 20) | (im > 235)).mean())
        c.append((score, im))
    if not c:
        return None
    c.sort(key=lambda z: -z[0])
    return c[0][1]


def square_cut(im, cy, cx, side, interp, out=S):
    """square crop centred at (cy,cx) with zero-padding beyond the image edge"""
    s  = int(round(side))
    y0 = int(round(cy - s / 2.0)); x0 = int(round(cx - s / 2.0))
    y1, x1 = y0 + s, x0 + s
    ty0, tx0 = max(0, -y0), max(0, -x0)
    ty1, tx1 = max(0, y1 - im.shape[0]), max(0, x1 - im.shape[1])
    sub = im[max(y0, 0):min(y1, im.shape[0]), max(x0, 0):min(x1, im.shape[1])]
    if sub.size == 0:
        return None
    if ty0 or tx0 or ty1 or tx1:
        sub = cv2.copyMakeBorder(sub, ty0, ty1, tx0, tx1, cv2.BORDER_CONSTANT, value=0)
    return cv2.resize(sub, (out, out), interpolation=interp)


rows, skip = [], {}
diag = {"ratio": [], "cov_old": [], "cov_new": [], "side": []}
t0 = time.time()

for i, r in d.iterrows():
    if (i + 1) % 200 == 0:
        print("   %4d/%d   (%.0fs)" % (i + 1, len(d), time.time() - t0), flush=True)

    old_m = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    if old_m is None:
        skip["no existing mask"] = skip.get("no existing mask", 0) + 1; continue
    cov_old = float((old_m > 127).mean())
    if cov_old < 1e-4:
        skip["empty existing mask"] = skip.get("empty existing mask", 0) + 1; continue

    full = read_full(r["full_series"])
    if full is None:
        skip["no full jpg"] = skip.get("no full jpg", 0) + 1; continue
    fm = read_maskseries(r["mask_series"], full.shape)
    if fm is None:
        skip["no mask jpg"] = skip.get("no mask jpg", 0) + 1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)

    ys, xs = np.where(fm > 127)
    if len(ys) < 20:
        skip["mask too small"] = skip.get("mask too small", 0) + 1; continue
    A  = float(len(ys))
    cy, cx = 0.5 * (ys.min() + ys.max()), 0.5 * (xs.min() + xs.max())
    bmax = float(max(ys.max() - ys.min(), xs.max() - xs.min()) + 1)

    side_tight = float(np.sqrt(A / cov_old))
    side_tight = float(np.clip(side_tight, 1.1 * bmax, 5.0 * bmax))
    side_wide  = side_tight * WIDE_MULT

    img_w = square_cut(full, cy, cx, side_wide, cv2.INTER_AREA)
    if img_w is None:
        skip["cut failed"] = skip.get("cut failed", 0) + 1; continue

    # ---- project the PREDICTED (official-split) mask into the wide frame ----
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    if pm is None:
        skip["no predmask"] = skip.get("no predmask", 0) + 1; continue
    st  = max(int(round(side_tight)), 8)
    pmb = cv2.resize((pm > 127).astype(np.uint8) * 255, (st, st),
                     interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros(full.shape, np.uint8)
    ty0 = int(round(cy - st / 2.0)); tx0 = int(round(cx - st / 2.0))
    sy0, sx0 = max(ty0, 0), max(tx0, 0)
    sy1, sx1 = min(ty0 + st, full.shape[0]), min(tx0 + st, full.shape[1])
    if sy1 > sy0 and sx1 > sx0:
        canvas[sy0:sy1, sx0:sx1] = pmb[sy0 - ty0:sy1 - ty0, sx0 - tx0:sx1 - tx0]
    msk_w = square_cut(canvas, cy, cx, side_wide, cv2.INTER_NEAREST)
    if msk_w is None:
        skip["cut failed"] = skip.get("cut failed", 0) + 1; continue

    stem = os.path.basename(str(r["img"])).replace("_img.png", "")
    pi = os.path.join(OUT, stem + "_img.png")
    pp = os.path.join(OUT, stem + "_pred.png")
    cv2.imwrite(pi, img_w)
    cv2.imwrite(pp, msk_w)

    gtw = square_cut(fm, cy, cx, side_wide, cv2.INTER_NEAREST)
    diag["ratio"].append(side_tight / bmax)
    diag["cov_old"].append(cov_old)
    diag["cov_new"].append(float((gtw > 127).mean()) if gtw is not None else np.nan)
    diag["side"].append(side_wide)
    rows.append((i, pi, pp))

print("\ngenerated %d / %d   (%.1f min)" % (len(rows), len(d), (time.time() - t0) / 60))
if skip:
    print("skipped:", skip)
assert len(rows) == len(d), \
    "only %d of %d wide crops were produced — CELL D/H assert on completeness" % (len(rows), len(d))

ok = pd.DataFrame(diag)
print("\n" + "=" * 66)
print("CROP GEOMETRY CHECK")
print("=" * 66)
print("  solved crop width / lesion width     median %.2f   (expect ~1.5-2.0)"
      % np.median(ok["ratio"]))
print("  lesion coverage, tight crops         median %.1f%%" % (100 * np.median(ok["cov_old"])))
print("  lesion coverage, WIDE crops          median %.1f%%   (expect ~%.1f%%)"
      % (100 * np.nanmedian(ok["cov_new"]), 100 * np.median(ok["cov_old"]) / WIDE_MULT ** 2))
print("  native pixels of the wide window     median %.0f" % np.median(ok["side"]))
print("  fraction of wide windows >= 512 px   %.1f%%"
      % (100 * float((np.array(ok["side"]) >= S).mean())))

sel = [r_[0] for r_ in rows[:: max(1, len(rows) // 4)]][:4]
fig, ax = plt.subplots(2, len(sel), figsize=(4 * len(sel), 8))
ax = np.atleast_2d(ax)
for j, gi in enumerate(sel):
    o  = cv2.imread(str(d.iloc[gi]["img"]), cv2.IMREAD_GRAYSCALE)
    om = cv2.imread(str(d.iloc[gi]["predmask"]), cv2.IMREAD_GRAYSCALE)
    stem = os.path.basename(str(d.iloc[gi]["img"])).replace("_img.png", "")
    w  = cv2.imread(os.path.join(OUT, stem + "_img.png"), cv2.IMREAD_GRAYSCALE)
    wm = cv2.imread(os.path.join(OUT, stem + "_pred.png"), cv2.IMREAD_GRAYSCALE)
    for row, (im_, m_, ttl) in enumerate([(o, om, "tight crop"), (w, wm, "wide context")]):
        ax[row, j].imshow(im_, cmap="gray")
        if m_ is not None and (m_ > 127).any():
            ax[row, j].contour(m_ > 127, levels=[0.5], colors="lime", linewidths=1.4)
        ax[row, j].set_title("%s\n%s" % (ttl, "MALIGNANT" if d.iloc[gi]["label"] == 1 else "benign"),
                             fontsize=9)
        ax[row, j].axis("off")
plt.tight_layout()
fp = os.path.join(FIG, "widectx_check_official.png")
plt.savefig(fp, dpi=130, bbox_inches="tight"); plt.close()
print("\n  figure: %s" % fp)

keep = pd.DataFrame(rows, columns=["ridx", "wide_img", "wide_pred"]).set_index("ridx")
nd = d.loc[keep.index].copy()
nd["img"]      = keep["wide_img"].values
nd["predmask"] = keep["wide_pred"].values
out_csv = os.path.join(D, "unified_folds_%s_wide_official.csv" % LES)
nd.to_csv(out_csv, index=False)
print("  saved  %s   (%d rows, folds/roles preserved)" % (os.path.basename(out_csv), len(nd)))
print("\n  -> next: CELL 6-OFFICIAL v3, then CELL D v2 with the _official paths.")

VERIFICATION
  mask source        : /root/autodl-tmp/CBIS/predmasks_mass_official
  official masks     : 1696 / 1696 present
  official split     : train 1318 | test 378 regions
  patients in BOTH   : 0
  output folder      : /root/autodl-tmp/CBIS/crops_wide_mass_official
  (this cell only re-crops; the masks it projects are PREDICTED, never ground truth)

    200/1696   (24s)
    400/1696   (49s)
    600/1696   (72s)
    800/1696   (97s)
   1000/1696   (121s)
   1200/1696   (145s)
   1400/1696   (170s)
   1600/1696   (193s)

generated 1696 / 1696   (3.4 min)

CROP GEOMETRY CHECK
  solved crop width / lesion width     median 1.57   (expect ~1.5-2.0)
  lesion coverage, tight crops         median 23.0%
  lesion coverage, WIDE crops          median 7.5%   (expect ~7.5%)
  native pixels of the wide window     median 882
  fraction of wide windows >= 512 px   94.5%

  figure: /root/autodl-tmp/CBIS/figures/mass_final/widectx_check_official.png
  saved  unified_folds_mass_wide_official.csv   

## C · Classifiers


**`SEG` cell 9** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6-OFFICIAL v3 — MASS CLASSIFIER (DenseNet-121, mask-weighted dual
# pooling) on the OFFICIAL SPLIT, using the OFFICIAL-SPLIT MASKS.
#   Identical hyperparameters to your CV cell 19.
#   Resumable per fold. Saves test-side AND train-side predictions.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

TAG  = "v2_offmask"
CKPT = os.path.join(D, "ckpt_official"); os.makedirs(CKPT, exist_ok=True)
PM   = os.path.join(D, "predmasks_%s_official" % LES)      # <-- OFFICIAL masks

S, BATCH   = 512, 12
SEEDS      = [11, 22]
EPOCHS     = 22
FREEZE     = 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W = 1e-4, 2.0, 0.3
MULT, PATIENCE, ATT = 4, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
miss = (~d["predmask"].apply(os.path.exists)).sum()
assert miss == 0, "%d official masks missing in %s — run the segmentation cell first" % (miss, PM)
assert "role_of0" in d.columns, "run CELL A first"
_te = d["official_split"].astype(str).str.lower().str.contains("test")
for k in range(5):
    r = d["role_of%d" % k]
    assert ((r == "test") == _te).all(), "role_of%d test != official test" % k
    assert not (r.isin(["train","val"]) & _te).any(), "training on official TEST!"
print("masks   : %s" % PM)
print("OFFICIAL SPLIT | train %d | test %d regions | %d test lesions"
      % ((~_te).sum(), _te.sum(), d.loc[_te, "lesion_key"].nunique()))

def ck_path(f): return os.path.join(CKPT, "%s_f%d.npz" % (TAG, f))
def ck_save(f, te, pt, va, pv):
    np.savez(ck_path(f),
             te_img=d.iloc[te]["img"].values.astype(str), te_prob=np.asarray(pt, np.float64),
             va_img=d.iloc[va]["img"].values.astype(str), va_prob=np.asarray(pv, np.float64))
def ck_load(f, te):
    p = ck_path(f)
    if not os.path.exists(p): return None
    try:
        z = np.load(p, allow_pickle=True)
        if list(z["te_img"]) != list(d.iloc[te]["img"].values.astype(str)):
            print("  (stale checkpoint fold %d ignored)" % f); return None
        return z
    except Exception as e:
        print("  (unreadable checkpoint fold %d: %s)" % (f, str(e)[:40])); return None
print("checkpoints on disk: %s" % [f for f in range(5) if os.path.exists(ck_path(f))])

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    k = str(r["img"])
    if k in CACHE: continue
    im = cv2.imread(k, cv2.IMREAD_GRAYSCALE)
    im = np.zeros((S, S), np.uint8) if im is None else im
    if im.shape != (S, S): im = cv2.resize(im, (S, S))
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    pm = (np.zeros((S, S), np.uint8) if pm is None
          else (cv2.resize(pm, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    CACHE[k] = (_clahe.apply(im), pm)
print("cached %d images in %.0fs" % (len(CACHE), time.time() - t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx=np.asarray(idx); self.aug=aug; self.mult=mult if aug else 1; self.tta=tta
    def __len__(self): return len(self.idx)*self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        img, msk = CACHE[str(r["img"])]; img, msk = img.copy(), msk.copy()
        if self.aug:
            if np.random.rand() < 0.5: img, msk = img[:, ::-1], msk[:, ::-1]
            if np.random.rand() < 0.5: img, msk = img[::-1, :], msk[::-1, :]
            k = np.random.randint(4)
            if k: img, msk = np.rot90(img, k), np.rot90(msk, k)
            img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
            if np.random.rand() < 0.7:
                M = cv2.getRotationMatrix2D((S/2, S/2), np.random.uniform(-25,25),
                                            np.random.uniform(0.90,1.12))
                img = cv2.warpAffine(img, M, (S,S), flags=cv2.INTER_LINEAR,
                                     borderMode=cv2.BORDER_REFLECT)
                msk = cv2.warpAffine(msk, M, (S,S), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT)
            if np.random.rand() < 0.5:
                img = np.clip(img.astype(np.float32)*np.random.uniform(0.85,1.15)
                              + np.random.uniform(-12,12), 0, 255).astype(np.uint8)
        else:
            t = self.tta
            if   t == 1: img, msk = img[:, ::-1], msk[:, ::-1]
            elif t == 2: img, msk = img[::-1, :], msk[::-1, :]
            elif t == 3: img, msk = np.rot90(img, 2), np.rot90(msk, 2)
        img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
        im = img.astype(np.float32)/255.0
        x = ((np.stack([im, im, im], 0) - MEAN)/STD).astype(np.float32)
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return (torch.from_numpy(x), torch.from_numpy(msk.astype(np.float32))[None],
                torch.tensor(int(r["label"])), torch.from_numpy(av))

class GuidedNet(nn.Module):
    """DenseNet-121 + mask-weighted pooling AND unweighted whole-crop pooling."""
    def __init__(self, aux_meta, att=2.0):
        super().__init__()
        try: dn = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception:
            dn = models.densenet121(weights=None); print("  (ImageNet weights unavailable)")
        self.b, self.att = dn.features, att
        Fdim = 1024
        self.head = nn.Sequential(nn.Linear(Fdim*2, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(aux_meta.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fdim*2,128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, aux_meta[k]))
                                  for k in self.keys])
    def forward(self, x, mask):
        f = F.relu(self.b(x))
        m = F.interpolate(mask, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att * m
        g_les = (f*w).sum((2,3)) / (w.sum((2,3)) + 1e-6)
        g_all = f.mean((2,3))
        g = torch.cat([g_les, g_all], 1)
        return self.head(g), [h(g) for h in self.aux]

def focal(logits, target, alpha):
    ce = F.cross_entropy(logits.float(), target, weight=alpha, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0,1,2,3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=20, shuffle=False, num_workers=0)
        ps = []
        for x, m, _, _ in ld:
            x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o, _ = net(x, m)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    y = d["label"].values
    n0, n1 = float((y[tr]==0).sum()), float((y[tr]==1).sum())
    alpha = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = GuidedNet(meta, ATT).to(DEV).to(memory_format=torch.channels_last)
    for p in net.b.parameters(): p.requires_grad = False
    head_params = [p for n_, p in net.named_parameters() if not n_.startswith("b.")]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(head_params, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, best_state, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        if ep == FREEZE + 1:
            for p in net.b.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.b.parameters(), "lr": LR_BACK},
                                     {"params": head_params,        "lr": LR_HEAD_FT}],
                                    weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.b.eval()
        for x, m, t, a in tl:
            x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            m = m.to(DEV, non_blocking=True); t = t.to(DEV); a = a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(x, m)
                loss = focal(o, t, alpha)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                        for h, g in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            star = " *"
        else: bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star))
        if bad >= PATIENCE: print("      early stop"); break
    net.load_state_dict({k: v.to(DEV) for k, v in best_state.items()})
    pt = predict(net, te, tta=True)
    pv = predict(net, va, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, pv, best

y = d["label"].values
for k in FOLDS:
    role = d["role_of%d" % k]
    tr = np.where(role=="train")[0]; va = np.where(role=="val")[0]; te = np.where(role=="test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK train/test"
    assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "LEAK val/test"
    if ck_load(k, te) is not None:
        print("\n### fold %d — already on disk, skipping" % k); continue
    print("\n### fold %d | train %d | val %d | test %d" % (k, len(tr), len(va), len(te)))
    pts, pvs, t0 = [], [], time.time()
    for sd in SEEDS:
        print("    seed %d" % sd)
        p, v, bv = train_one(tr, va, te, sd)
        pts.append(p); pvs.append(v)
        print("    seed %d done: best val %.4f | test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p)))
    ck_save(k, te, np.mean(pts, 0), va, np.mean(pvs, 0))
    print("  FOLD %d this fold AUC %.4f | CHECKPOINT SAVED (%.0fs)"
          % (k, roc_auc_score(y[te], np.mean(pts, 0)), time.time()-t0))

te0 = np.where(d["role_of0"].values == "test")[0]
parts, va_img, va_prob, have = [], [], [], []
for k in range(5):
    te = np.where(d["role_of%d" % k].values == "test")[0]
    z = ck_load(k, te)
    if z is None: continue
    parts.append(z["te_prob"]); have.append(k)
    va_img += list(z["va_img"]); va_prob += list(z["va_prob"])

print("\n" + "="*70)
print("MASS CLASSIFIER v2 (official masks)  |  folds completed: %s (%d/5)" % (have, len(have)))
print("="*70)
if not parts: raise SystemExit("no checkpoints yet")
oof = np.mean(parts, axis=0)
for i, f in enumerate(have):
    print("     after fold %d  ->  %.4f" % (f, roc_auc_score(y[te0], np.mean(parts[:i+1], axis=0))))
res = d.iloc[te0][["img","lesion_key","label"]].copy(); res["prob"] = oof
L = res.groupby("lesion_key").agg(y=("label","max"), p=("prob","mean")).reset_index()
grid = np.linspace(0.05, 0.95, 181)
thr = grid[int(np.argmax([balanced_accuracy_score(L.y, (L.p > t).astype(int)) for t in grid]))]
pr = (L.p > thr).astype(int)
tn, fp, fn, tp = confusion_matrix(L.y, pr, labels=[0,1]).ravel()
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(y[te0], oof), len(te0)))
print("  per-lesion AUC %.4f  acc %.1f%%  sens %.3f  spec %.3f  FP %d FN %d"
      % (roc_auc_score(L.y, L.p), 100*accuracy_score(L.y, pr),
         tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))
print("  previous run with CV masks: 0.8652")

res.rename(columns={"label":"true"}).to_csv(
    os.path.join(D, "cv_mass_officialsplit_oof.csv"), index=False)
if va_img:
    tr_ = pd.DataFrame({"img": va_img, "prob": va_prob})
    tr_ = tr_.merge(d[["img","lesion_key","label"]], on="img", how="left").rename(columns={"label":"true"})
    tr_.to_csv(os.path.join(D, "cv_mass_v2_officialtrain_oof.csv"), index=False)
    print("  saved train-side file (%d rows)" % len(tr_))
print("  -> next: CELL D v2 with the official masks")
if len(have) < 5: print("  PARTIAL — re-run this cell later to add the missing folds.")

masks   : /root/autodl-tmp/CBIS/predmasks_mass_official
OFFICIAL SPLIT | train 1318 | test 378 regions | 223 test lesions
checkpoints on disk: []
cached 1696 images in 8s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1054 | val 264 | test 378
    seed 11
      ep  1  val-AUC 0.7407 *
      ep  2  val-AUC 0.5810
      ep  3  val-AUC 0.6325
      ep  4  val-AUC 0.7621 *
      ep  5  val-AUC 0.7877 *
      ep  6  val-AUC 0.8379 *
      ep  7  val-AUC 0.8567 *
      ep  8  val-AUC 0.8278
      ep  9  val-AUC 0.8603 *
      ep 10  val-AUC 0.8728 *
      ep 11  val-AUC 0.8534
      ep 12  val-AUC 0.8394
      ep 13  val-AUC 0.8689
      ep 14  val-AUC 0.8611
      ep 15  val-AUC 0.8606
      ep 16  val-AUC 0.8578
      ep 17  val-AUC 0.8592
      early stop
    seed 11 done: best val 0.8728 | test AUC 0.8398
    seed 22
      ep  1  val-AUC 0.7410 *
      ep  2  val-AUC 0.7246
      ep  3  val-AUC 0.7378
      ep  4  val-AUC 0.7750 *
      ep  5  val-A

**`SEG` cell 10** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL D v2 — TWO-STREAM on the OFFICIAL SPLIT, OFFICIAL MASKS  [RESUMABLE]
#   Stream A: 512px tight crop + tight predicted mask
#   Stream B: 384px wide crop  + wide predicted mask
#   Architecture, seed and hyperparameters identical to your CV cell 47.
#   Saves test-side AND train-side predictions. Checkpoints every fold.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True

TAG  = "twostream_offmask"                                  # <- new tag: forces a fresh run
CKPT = os.path.join(D, "ckpt_official"); os.makedirs(CKPT, exist_ok=True)
WIDE = os.path.join(D, "crops_wide_%s_official" % LES)      # <- OFFICIAL wide crops
PM   = os.path.join(D, "predmasks_%s_official" % LES)       # <- OFFICIAL tight masks

ST, SW, BATCH = 512, 384, 8
SEEDS, EPOCHS, FREEZE = [11], 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png", ""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s + "_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_pred.png"))

print("=" * 70); print("VERIFICATION"); print("=" * 70)
print("  tight masks : %s" % PM)
print("  wide crops  : %s" % WIDE)
for col, name in [("tmask", "tight masks"), ("wimg", "wide images"), ("wmask", "wide masks")]:
    n = int(d[col].apply(os.path.exists).sum())
    print("  %-12s: %d / %d present" % (name, n, len(d)))
    assert n == len(d), "%d %s missing — run the segmentation cell and CELL 20A-OFFICIAL first" % (len(d)-n, name)
assert "role_of0" in d.columns, "run CELL A first"
_te = d["official_split"].astype(str).str.lower().str.contains("test")
for k in range(5):
    r = d["role_of%d" % k]
    assert ((r == "test") == _te).all(), "role_of%d test != official test" % k
    assert not (r.isin(["train","val"]) & _te).any(), "training on official TEST!"
print("  official split: train %d | test %d regions | %d test lesions"
      % ((~_te).sum(), _te.sum(), d.loc[_te, "lesion_key"].nunique()))
print("  patients in BOTH: %d" % len(set(d.patient_id[~_te]) & set(d.patient_id[_te])))
print("=" * 70)

def ck_path(f): return os.path.join(CKPT, "%s_f%d.npz" % (TAG, f))
def ck_save(f, te, pt, va, pv):
    np.savez(ck_path(f),
             te_img=d.iloc[te]["img"].values.astype(str), te_prob=np.asarray(pt, np.float64),
             va_img=d.iloc[va]["img"].values.astype(str), va_prob=np.asarray(pv, np.float64))
def ck_load(f, te):
    p = ck_path(f)
    if not os.path.exists(p): return None
    try:
        z = np.load(p, allow_pickle=True)
        if list(z["te_img"]) != list(d.iloc[te]["img"].values.astype(str)):
            print("  (stale checkpoint fold %d ignored)" % f); return None
        return z
    except Exception as e:
        print("  (unreadable checkpoint fold %d: %s)" % (f, str(e)[:40])); return None
print("checkpoints on disk: %s   (expect [] for a fresh run)"
      % [f for f in range(5) if os.path.exists(ck_path(f))])

cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size, size), np.uint8)
    if im.shape != (size, size):
        im = cv2.resize(im, (size, size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im > 127).astype(np.uint8) if mask else cl.apply(im)

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("cached %d in %.0fs" % (len(CACHE), time.time() - t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug = np.asarray(idx), aug
        self.mult, self.tta = (mult if aug else 1), tta
    def __len__(self): return len(self.idx) * self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand() < .5, np.random.rand() < .5
            kk = np.random.randint(4)
            aff = np.random.rand() < .7
            ang, sc = np.random.uniform(-25, 25), np.random.uniform(.9, 1.12)
            itn = np.random.rand() < .5
            gg, bb = np.random.uniform(.85, 1.15), np.random.uniform(-12, 12)
            def T(im, mk, s):
                if fh: im, mk = im[:, ::-1], mk[:, ::-1]
                if fv: im, mk = im[::-1, :], mk[::-1, :]
                if kk: im, mk = np.rot90(im, kk), np.rot90(mk, kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2, s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s, s), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s, s), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32)*gg + bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta
            def V(im, mk):
                if   t == 1: return im[:, ::-1], mk[:, ::-1]
                elif t == 2: return im[::-1, :], mk[::-1, :]
                elif t == 3: return np.rot90(im, 2), np.rot90(mk, 2)
                return im, mk
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        out = []
        for im, mk in [(ti, tm), (wi, wm)]:
            im = np.ascontiguousarray(im).astype(np.float32) / 255.0
            out.append(torch.from_numpy(((np.stack([im]*3, 0) - MEAN) / STD).astype(np.float32)))
            out.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return out[0], out[1], out[2], out[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, att=2.0):
        super().__init__()
        def bb():
            try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
            except Exception: return models.densenet121(weights=None).features
        self.bt, self.bw, self.att = bb(), bb(), att
        Fd = 1024 * 4
        self.head = nn.Sequential(nn.Linear(Fd, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd, 128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[k]))
                                  for k in self.keys])
    def pool(self, b, x, m):
        f = F.relu(b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att * mm
        return torch.cat([(f*w).sum((2,3)) / (w.sum((2,3)) + 1e-6), f.mean((2,3))], 1)
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([self.pool(self.bt, xt, mt), self.pool(self.bw, xw, mw)], 1)
        return self.head(g), [h(g) for h in self.aux]

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=12, shuffle=False, num_workers=0)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            xt, mt, xw, mw = xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o, _ = net(xt, mt, xw, mw)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    yy = d["label"].values
    n0, n1 = float((yy[tr] == 0).sum()), float((yy[tr] == 1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = TwoStream(meta, ATT).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for p_ in back: p_.requires_grad = False
    hp = [p_ for n_, p_ in net.named_parameters()
          if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p_ in back: p_.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS - FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, mt, xw, mw, t_, a_ in tl:
            xt, mt = xt.to(DEV, non_blocking=True), mt.to(DEV, non_blocking=True)
            xw, mw = xw.to(DEV, non_blocking=True), mw.to(DEV, non_blocking=True)
            t_, a_ = t_.to(DEV), a_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(gg.float(), a_[:, h], ignore_index=-1)
                        for h, gg in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(yy[va], pv) if len(set(yy[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            star = " *"
        else: bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break
    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, tta=True)
    pv = predict(net, va, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, pv, best

y = d["label"].values
for k in FOLDS:
    role = d["role_of%d" % k]
    tr = np.where(role == "train")[0]; va = np.where(role == "val")[0]; te = np.where(role == "test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK train/test"
    assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "LEAK val/test"
    if ck_load(k, te) is not None:
        print("\n### fold %d — already on disk, skipping" % k); continue
    print("\n### fold %d | train %d val %d test %d" % (k, len(tr), len(va), len(te)))
    pts, pvs, t0 = [], [], time.time()
    for sd in SEEDS:
        p_, v_, bv = train_one(tr, va, te, sd)
        pts.append(p_); pvs.append(v_)
        print("    seed %d: best val %.4f | test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p_)))
    ck_save(k, te, np.mean(pts, 0), va, np.mean(pvs, 0))
    print("  FOLD %d this fold AUC %.4f | CHECKPOINT SAVED (%.0fs)"
          % (k, roc_auc_score(y[te], np.mean(pts, 0)), time.time() - t0))

te0 = np.where(d["role_of0"].values == "test")[0]
parts, va_img, va_prob, have = [], [], [], []
for k in range(5):
    te = np.where(d["role_of%d" % k].values == "test")[0]
    z = ck_load(k, te)
    if z is None: continue
    parts.append(z["te_prob"]); have.append(k)
    va_img += list(z["va_img"]); va_prob += list(z["va_prob"])

print("\n" + "=" * 70)
print("TWO-STREAM (official masks)  |  folds completed: %s (%d/5)" % (have, len(have)))
print("=" * 70)
if not parts: raise SystemExit("no checkpoints yet")
oof = np.mean(parts, axis=0)
for i, f in enumerate(have):
    print("     after fold %d  ->  %.4f" % (f, roc_auc_score(y[te0], np.mean(parts[:i+1], axis=0))))
res = d.iloc[te0][["img", "lesion_key", "label"]].copy(); res["prob"] = oof
L = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean")).reset_index()
grid = np.linspace(0.05, 0.95, 181)
thr = grid[int(np.argmax([balanced_accuracy_score(L.y, (L.p > t).astype(int)) for t in grid]))]
pr = (L.p > thr).astype(int)
tn, fp, fn, tp = confusion_matrix(L.y, pr, labels=[0, 1]).ravel()
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(y[te0], oof), len(te0)))
print("  per-lesion AUC %.4f  acc %.1f%%  sens %.3f  spec %.3f  FP %d FN %d"
      % (roc_auc_score(L.y, L.p), 100*accuracy_score(L.y, pr),
         tp/max(tp+fn, 1), tn/max(tn+fp, 1), fp, fn))
print("  previous run with CV masks: 0.9004")
print("  v2 with official masks:     0.8715")

res.rename(columns={"label": "true"}).to_csv(
    os.path.join(D, "cv_mass_twostream_officialsplit_oof.csv"), index=False)
if va_img:
    tr_ = pd.DataFrame({"img": va_img, "prob": va_prob})
    tr_ = tr_.merge(d[["img", "lesion_key", "label"]], on="img", how="left").rename(columns={"label": "true"})
    tr_.to_csv(os.path.join(D, "cv_mass_twostream_officialtrain_oof.csv"), index=False)
    print("  saved train-side file (%d rows)" % len(tr_))
print("  -> next: CELL H (calcpre), then CELL E v4")
if len(have) < 5: print("  PARTIAL — re-run this cell later to add the missing folds.")

VERIFICATION
  tight masks : /root/autodl-tmp/CBIS/predmasks_mass_official
  wide crops  : /root/autodl-tmp/CBIS/crops_wide_mass_official
  tight masks : 1696 / 1696 present
  wide images : 1696 / 1696 present
  wide masks  : 1696 / 1696 present
  official split: train 1318 | test 378 regions | 223 test lesions
  patients in BOTH: 0
checkpoints on disk: []   (expect [] for a fresh run)
cached 1696 in 17s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1054 val 264 test 378
      ep  1  val-AUC 0.7138 *
      ep  2  val-AUC 0.6447
      ep  3  val-AUC 0.6282
      ep  4  val-AUC 0.8085 *
      ep  5  val-AUC 0.8697 *
      ep  6  val-AUC 0.8760 *
      ep  7  val-AUC 0.8872 *
      ep  8  val-AUC 0.9039 *
      ep  9  val-AUC 0.8805
      ep 10  val-AUC 0.9057 *
      ep 11  val-AUC 0.8904
      ep 12  val-AUC 0.9001
      ep 13  val-AUC 0.8948
      ep 14  val-AUC 0.8889
      ep 15  val-AUC 0.9061 *
      ep 16  val-AUC 0.8973
      ep 17  val-AUC

**`IL1` cell 0** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# TWO-STREAM v2  —  higher resolution, cosine+warmup, EMA, 8-way TTA
#   MODE = "cv"       -> patient-grouped 5-fold      (role_f0..4)
#   MODE = "official" -> official TCIA split         (role_of0..4)
#   Architecture UNCHANGED: mask-weighted pooling + wide stream + aux heads.
#   Every (fold, seed) checkpointed to .npz - a crash costs one fold.
#   >>> RUN WITH QUICK_TEST = True FIRST <<<
# ══════════════════════════════════════════════════════════════════════
import os, gc, time, math, json
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ─────────────────────────── CONFIG ───────────────────────────
QUICK_TEST = False          # <<< True first. One fold, 3 epochs, error check only.
MODE       = "cv"          # "cv" or "official"
TAG        = "twostream_v2"

ST, SW     = 640, 448      # was 512 / 384   <- main gain. Drop to 576/416 if OOM.
BATCH, ACC = 6, 2          # effective batch 12. Lower BATCH if OOM.
SEEDS      = [11, 22]      # [11] for a single-seed run
EPOCHS, FREEZE, WARMUP = 26, 3, 2
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 8e-5    # LR_BACK was 3e-5 - too low
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 8, 2.0
EMA_DECAY  = 0.999
TTA_N      = 8             # was 4
FOLDS      = [0, 1, 2, 3, 4]
USE_AMP    = True
# ──────────────────────────────────────────────────────────────

if QUICK_TEST:
    SEEDS, EPOCHS, FREEZE, WARMUP, MULT, FOLDS = [11], 3, 1, 1, 1, [0]
    print(">>> QUICK TEST: 1 fold, 3 epochs, no checkpoint written\n")

ROLE = "role_f" if MODE == "cv" else "role_of"
CKPT = os.path.join(D, "ckpt_%s_%s" % (TAG, MODE)); os.makedirs(CKPT, exist_ok=True)

WIDE = os.path.join(D, "crops_wide_%s_official" % LES)
if not os.path.isdir(WIDE): WIDE = os.path.join(D, "crops_wide_%s" % LES)
PM   = os.path.join(D, "predmasks_%s_official" % LES)
if not os.path.isdir(PM):   PM = os.path.join(D, "predmasks_%s" % LES)
print("wide crops : %s\npred masks : %s" % (WIDE, PM))

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png", ""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s + "_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_pred.png"))
assert (~d["wimg"].apply(os.path.exists)).sum() == 0, "wide crops missing"
assert "%s0" % ROLE in d.columns, "%s0 column missing" % ROLE

if MODE == "official":
    _te = d["official_split"].astype(str).str.lower().str.contains("test")
    for _k in range(5):
        assert ((d["%s%d" % (ROLE, _k)] == "test") == _te).all(), "fold %d != official test" % _k
    print("verified OFFICIAL SPLIT | train %d | test %d" % ((~_te).sum(), _te.sum()))
else:
    print("patient-grouped 5-fold CV | %d regions | %d patients"
          % (len(d), d.patient_id.nunique()))
print("malignant %.1f%%\n" % (100 * d.label.mean()))

# ─────────────────── cache (RAM ~2 GB at 640/448) ───────────────────
cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size, size), np.uint8)
    if im.shape != (size, size):
        im = cv2.resize(im, (size, size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im > 127).astype(np.uint8) if mask else cl.apply(im)

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("cached %d crops in %.0fs (~%.1f GB)"
      % (len(CACHE), time.time() - t0, len(CACHE) * (2*ST*ST + 2*SW*SW) / 1e9))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"]); mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta, "\n")

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug = np.asarray(idx), aug
        self.mult, self.tta = (mult if aug else 1), tta
    def __len__(self): return len(self.idx) * self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand() < .5, np.random.rand() < .5
            kk = np.random.randint(4)
            aff = np.random.rand() < .7
            ang, sc = np.random.uniform(-25, 25), np.random.uniform(.88, 1.14)
            itn = np.random.rand() < .5
            gg, bb = np.random.uniform(.85, 1.15), np.random.uniform(-12, 12)
            def T(im, mk, s):
                if fh: im, mk = im[:, ::-1], mk[:, ::-1]
                if fv: im, mk = im[::-1, :], mk[::-1, :]
                if kk: im, mk = np.rot90(im, kk), np.rot90(mk, kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2, s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s, s), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s, s), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32) * gg + bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta; rot, flip = t % 4, t // 4      # 8 dihedral transforms
            def V(im, mk):
                if rot: im, mk = np.rot90(im, rot), np.rot90(mk, rot)
                if flip: im, mk = im[:, ::-1], mk[:, ::-1]
                return np.ascontiguousarray(im), np.ascontiguousarray(mk)
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        out = []
        for im, mk in [(ti, tm), (wi, wm)]:
            im = np.ascontiguousarray(im).astype(np.float32) / 255.0
            out.append(torch.from_numpy(((np.stack([im]*3, 0) - MEAN) / STD).astype(np.float32)))
            out.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64)
        return out[0], out[1], out[2], out[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, att=2.0):
        super().__init__()
        def bb():
            try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
            except Exception: return models.densenet121(weights=None).features
        self.bt, self.bw, self.att = bb(), bb(), att
        Fd = 1024 * 4
        self.head = nn.Sequential(nn.Linear(Fd, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd, 128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[k]))
                                  for k in self.keys])
    def pool(self, b, x, m):
        f = F.relu(b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att * mm
        return torch.cat([(f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6), f.mean((2, 3))], 1)
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([self.pool(self.bt, xt, mt), self.pool(self.bw, xw, mw)], 1)
        return self.head(g), [h(g) for h in self.aux]

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone().float()
    def state(self, model):
        ref = model.state_dict()
        return {k: self.shadow[k].to(ref[k].dtype) for k in ref}

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, n_tta=1):
    net.eval(); tot = None
    for t in range(n_tta):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=max(4, BATCH),
                        shuffle=False, num_workers=0, pin_memory=True)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            xt, mt, xw, mw = xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV)
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                o, _ = net(xt, mt, xw, mw)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / n_tta

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    yy = d["label"].values
    n0, n1 = float((yy[tr] == 0).sum()), float((yy[tr] == 1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)

    net = TwoStream(meta, ATT).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for p_ in back: p_.requires_grad = False
    hp = [p_ for n_, p_ in net.named_parameters() if not (n_.startswith("bt.") or n_.startswith("bw."))]

    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD)
    sch, ema = None, None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, bstate, bad, skipped = -1.0, None, 0, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p_ in back: p_.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            T = max(1, EPOCHS - FREEZE)
            def lam(e):
                if e < WARMUP: return (e + 1) / WARMUP
                return 0.5 * (1 + math.cos(math.pi * (e - WARMUP) / max(1, T - WARMUP)))
            sch = torch.optim.lr_scheduler.LambdaLR(opt, lam)
            ema = EMA(net, EMA_DECAY)

        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        opt.zero_grad(set_to_none=True)
        for bi, (xt, mt, xw, mw, t_, a_) in enumerate(tl):
            xt, mt, xw, mw = (xt.to(DEV, non_blocking=True), mt.to(DEV, non_blocking=True),
                              xw.to(DEV, non_blocking=True), mw.to(DEV, non_blocking=True))
            t_, a_ = t_.to(DEV), a_.to(DEV)
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(gg.float(), a_[:, h], ignore_index=-1)
                        for h, gg in enumerate(ax)) / len(ax)
                loss = loss / ACC
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward()
            if (bi + 1) % ACC == 0:
                scaler.unscale_(opt)
                gn = torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                if torch.isfinite(gn): scaler.step(opt)
                else: skipped += 1
                scaler.update(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(net)
        if sch is not None: sch.step()

        # evaluate the EMA weights once they exist
        if ema is not None:
            raw = {k: v.detach().clone() for k, v in net.state_dict().items()}
            net.load_state_dict(ema.state(net))
        pv = predict(net, va, n_tta=1)
        if not np.all(np.isfinite(pv)):
            print("      ep %2d  NON-FINITE predictions - diverged" % ep)
            if bstate is not None: break
            raise RuntimeError("diverged; set USE_AMP = False and rerun")
        auc = roc_auc_score(yy[va], pv) if len(set(yy[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}; star = " *"
        else: bad += 1
        if ema is not None: net.load_state_dict(raw)
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break

    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, n_tta=TTA_N)
    if skipped: print("      (%d optimiser steps skipped on non-finite grads)" % skipped)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, best

# ─────────────────────────── run ───────────────────────────
y = d["label"].values
acc_p, cnt_p = np.zeros(len(d)), np.zeros(len(d))
oof = np.full(len(d), np.nan)

for k in FOLDS:
    role = d["%s%d" % (ROLE, k)]
    tr = np.where(role == "train")[0]; va = np.where(role == "val")[0]; te = np.where(role == "test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK train/test"
    assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "LEAK val/test"
    print("\n### fold %d | train %d val %d test %d" % (k, len(tr), len(va), len(te)))
    ps, t0 = [], time.time()
    for sd in SEEDS:
        ck = os.path.join(CKPT, "f%d_s%d.npz" % (k, sd))
        if os.path.exists(ck) and not QUICK_TEST:
            z = np.load(ck, allow_pickle=True)
            if list(z["te_idx"]) == list(te):
                ps.append(z["prob"]); print("    seed %d: loaded checkpoint (val %.4f)" % (sd, float(z["val"])))
                continue
            print("    seed %d: stale checkpoint, retraining" % sd)
        p_, bv = train_one(tr, va, te, sd); ps.append(p_)
        if not QUICK_TEST:
            np.savez(ck, prob=p_, val=bv, te_idx=np.array(te))
        print("    seed %d: best val %.4f | fold test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p_)))
    pmn = np.mean(ps, 0)
    acc_p[te] += pmn; cnt_p[te] += 1; oof[te] = acc_p[te] / cnt_p[te]
    print("  FOLD %d AUC %.4f | pooled so far %.4f (%.0fs)"
          % (k, roc_auc_score(y[te], pmn), roc_auc_score(y[te], oof[te]), time.time() - t0))

done = ~np.isnan(oof)
res = d.loc[done, ["img", "lesion_key", "label"]].copy(); res["prob"] = oof[done]
Lg = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean")).reset_index()
print("\n" + "=" * 70)
print("TWO-STREAM v2   MODE=%s   ST=%d SW=%d   seeds=%s" % (MODE, ST, SW, SEEDS))
print("=" * 70)
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(res.label, res.prob), len(res)))
print("  per-lesion AUC %.4f  (n=%d)" % (roc_auc_score(Lg.y, Lg.p), len(Lg)))
print("  reference: v1 official image 0.8769 / lesion 0.9043 | v1 CV image 0.8734 / lesion 0.8885")
if not QUICK_TEST:
    suffix = "officialsplit" if MODE == "official" else "cv"
    out = os.path.join(D, "cv_%s_%s_%s_oof.csv" % (LES, TAG, suffix))
    res.rename(columns={"label": "true"}).to_csv(out, index=False)
    print("\n  saved %s" % os.path.basename(out))
else:
    print("\n  QUICK TEST clean. Set QUICK_TEST = False and rerun.")

wide crops : /root/autodl-tmp/CBIS/crops_wide_mass_official
pred masks : /root/autodl-tmp/CBIS/predmasks_mass_official
patient-grouped 5-fold CV | 1696 regions | 892 patients
malignant 46.2%

cached 1696 crops in 20s (~2.1 GB)
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5} 


### fold 0 | train 1054 val 264 test 378
      ep  1  val-AUC 0.7412 *
      ep  2  val-AUC 0.7703 *
      ep  3  val-AUC 0.7283
      ep  4  val-AUC 0.7070
      ep  5  val-AUC 0.7485
      ep  6  val-AUC 0.7909 *
      ep  7  val-AUC 0.8279 *
      ep  8  val-AUC 0.8586 *
      ep  9  val-AUC 0.8797 *
      ep 10  val-AUC 0.8857 *
      ep 11  val-AUC 0.8916 *
      ep 12  val-AUC 0.9007 *
      ep 13  val-AUC 0.9056 *
      ep 14  val-AUC 0.9080 *
      ep 15  val-AUC 0.9084 *
      ep 16  val-AUC 0.9061
      ep 17  val-AUC 0.9002
      ep 18  val-AUC 0.8965
      ep 19  val-AUC 0.8942
      ep 20  val-AUC 0.8912
      ep 21  val-AUC 0.8883
      ep 22  val-AUC 0.8861
      ep 23  val-AUC 0.

## D · Final results and decision layer


**`IL2` cell 0** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 4 — RAISE THE SCORE, OFFICIAL SPLIT
#
#   PROTOCOL (unchanged, locked): fit on official TRAIN (1,318 img / 782 les),
#   report on official TEST (378 img / 223 les). Zero patient overlap.
#
#   Every candidate score is judged ONLY by its TRAIN AUC. The test set is
#   touched once, at the end, by the single winning candidate.
#   Tie-break rule: among candidates within 0.002 TRAIN AUC of the best,
#   take the SIMPLEST (fewest members). Guards against selection overfitting.
#
#   No GPU, no training. ~2 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, glob, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

D          = "/root/autodl-tmp/CBIS"
SENS_FLOOR = 0.70
GRID       = np.round(np.arange(0.02, 0.99, 0.01), 3)
MIN_N, MIN_POS, PASSES = 20, 3, 12
TIE, MAXK  = 0.002, 4
NBOOT, RNG = 2000, np.random.default_rng(7)
MATCH_SENS = [0.80, 0.85, 0.90, 0.95]

stem  = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
LOGIT = lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))
SIG   = lambda z: 1/(1+np.exp(-z))

# ─────────────────────────────────────────────────────────────────────────
# PART A — discover every (official-train, official-test) prediction pair
# ─────────────────────────────────────────────────────────────────────────
print("="*80); print("PART A — AVAILABLE MEMBERS (train scores are out-of-fold)"); print("="*80)

d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv"))
d["_k"] = d["img"].map(stem); assert d["_k"].is_unique
d["y"]  = d["label"].astype(int)
d["a"]  = pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"] = np.where(d["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
META = d.set_index("_k")[["y","a","lesion_key","patient_id","sp"]]

def load(fn):
    f = os.path.join(D, fn)
    if not os.path.exists(f): return None
    m = pd.read_csv(f)
    if "img" not in m.columns: return None
    pc = "prob" if "prob" in m.columns else None
    if pc is None:
        c = [x for x in m.columns if m[x].dtype.kind=="f" and m[x].between(0,1).all()]
        if not c: return None
        pc = c[0]
    m["_k"] = m["img"].map(stem)
    m = m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")
    return m[m["_k"].isin(META.index)]

MEM = {}
for trf in sorted(glob.glob(os.path.join(D, "*officialtrain*.csv"))):
    b   = os.path.basename(trf)
    tef = b.replace("officialtrain", "officialsplit")
    A, B = load(b), load(tef)
    if A is None or B is None or len(A) < 900 or len(B) < 300: continue
    nm = b.replace("cv_mass_","").replace("_officialtrain_oof","").replace(".csv","")
    ka = A.set_index("_k")["p"]; kb = B.set_index("_k")["p"]
    ya = META.loc[ka.index,"y"].values; yb = META.loc[kb.index,"y"].values
    if len(np.unique(ya))<2 or len(np.unique(yb))<2: continue
    atr, ate = roc_auc_score(ya, ka.values), roc_auc_score(yb, kb.values)
    if atr > 0.97: print("   SKIP %-34s TRAIN AUC %.4f (leakage guard)" % (nm, atr)); continue
    MEM[nm] = (ka, kb)
    print("   %-34s n_tr %4d n_te %3d   TRAIN AUC %.4f   test %.4f"
          % (nm, len(ka), len(kb), atr, ate))
assert MEM, "no paired official train/test prediction files found"

# ─────────────────────────────────────────────────────────────────────────
# PART B — is the two-stream file already seed-averaged?
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "="*80); print("PART B — SEED CHECK"); print("="*80)
one, avg = load("cv_mass_twostream_officialsplit_oof_1seed.csv"), load("cv_mass_twostream_officialsplit_oof.csv")
if one is not None and avg is not None:
    j = one.merge(avg, on="_k", suffixes=("_1","_m"))
    same = np.allclose(j.p_1.values, j.p_m.values, atol=1e-6)
    print("  officialsplit_oof vs _1seed : %s"
          % ("IDENTICAL -> current file is SINGLE seed, averaging is still available"
             if same else "DIFFERENT -> current file is already a seed average"))
    print("  corr %.5f | mean abs diff %.5f"
          % (np.corrcoef(j.p_1, j.p_m)[0,1], np.abs(j.p_1-j.p_m).mean()))
else:
    print("  one of the two files is missing; skipping")

s11, s22 = load("cv_mass_twostream_s11.csv"), load("cv_mass_twostream_s22.csv")
if s11 is not None and s22 is not None:
    j = s11.merge(s22, on="_k", suffixes=("_a","_b"))
    yy = META.loc[j._k,"y"].values
    print("  CV seeds: s11 AUC %.4f | s22 AUC %.4f | average %.4f  (%+.4f)"
          % (roc_auc_score(yy,j.p_a), roc_auc_score(yy,j.p_b),
             roc_auc_score(yy,(j.p_a+j.p_b)/2),
             roc_auc_score(yy,(j.p_a+j.p_b)/2)-max(roc_auc_score(yy,j.p_a),roc_auc_score(yy,j.p_b))))

# ─────────────────────────────────────────────────────────────────────────
# PART C — build candidates, judge them on TRAIN ONLY
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "="*80); print("PART C — CANDIDATE SELECTION (TRAIN AUC ONLY)"); print("="*80)

names = list(MEM)
ktr = sorted(set.intersection(*[set(MEM[n][0].index) for n in names]))
kte = sorted(set.intersection(*[set(MEM[n][1].index) for n in names]))
Ytr, Yte = META.loc[ktr,"y"].values, META.loc[kte,"y"].values
Ptr = {n: MEM[n][0].loc[ktr].values for n in names}
Pte = {n: MEM[n][1].loc[kte].values for n in names}
print("  common rows: train %d, test %d" % (len(ktr), len(kte)))

def mix(sel, P, how):
    M = np.column_stack([P[m] for m in sel])
    return M.mean(1) if how=="prob" else SIG(LOGIT(M).mean(1))

CAND = []
for n in names:                                   # singles
    CAND.append(((n,), "prob", roc_auc_score(Ytr, Ptr[n])))
for how in ("prob","logit"):                      # greedy forward, capped
    ch, best = [], -np.inf
    while len(ch) < MAXK:
        pk, pa = None, best
        for n in names:
            if n in ch: continue
            v = roc_auc_score(Ytr, mix(ch+[n], Ptr, how))
            if v > pa + 1e-6: pk, pa = n, v
        if pk is None: break
        ch.append(pk); best = pa
        CAND.append((tuple(ch), how, best))

CAND.sort(key=lambda c: -c[2])
top = CAND[0][2]
print("\n  top candidates by TRAIN AUC")
for sel, how, v in CAND[:8]:
    print("    %-6s %.4f  [%s]" % (how, v, " + ".join(sel)))
elig = [c for c in CAND if c[2] >= top - TIE]
SEL, HOW, SAUC = min(elig, key=lambda c: (len(c[0]), -c[2]))
print("\n  within %.3f of best: %d candidates -> take the simplest" % (TIE, len(elig)))
print("  CHOSEN: %s  [%s]   TRAIN AUC %.4f" % (HOW, " + ".join(SEL), SAUC))

# ─────────────────────────────────────────────────────────────────────────
# PART D — decision layer + final report on TEST
# ─────────────────────────────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,floor=SENS_FLOOR):
    P,N = int(y.sum()), len(y)
    tp,fp = _cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se = (tp+(N-P)-fp)/N, tp/max(P,1); ok = se>=floor-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[int(np.argmax(se))])
def _asc(y,p,a,init,floor):
    N,P = len(y), int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1) < floor-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=floor-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j] > (TP+TN)/N + 1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,floor=SENS_FLOOR):
    g0=fit_global(y,p,floor); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (0.50,g0,0.35,0.45,0.55,0.65,max(.02,g0-.1),min(.98,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.2,.8)) for g in cats} for _ in range(4)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,floor)
        if ac>ba+1e-12: best,ba=t,ac
    return best,g0
ap = lambda p,a,t,g0: (p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),
                sens=tp/max(tp+fn,1),spec=tn/max(tn+fp,1),fn=fn)
def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)), float(np.percentile(o,97.5))

BASE = "twostream" if "twostream" in names else names[0]
str_, ste_ = mix(SEL,Ptr,HOW), mix(SEL,Pte,HOW)
FINAL = {}
for level in ("IMAGE","LESION"):
    if level=="IMAGE":
        ytr,atr,ptr = Ytr, META.loc[ktr,"a"].values, str_
        yte,ate,pte = Yte, META.loc[kte,"a"].values, ste_
        btr,bte = Ptr[BASE], Pte[BASE]
        pat = META.loc[kte,"patient_id"].values
    else:
        def agg(keys,p):
            t=pd.DataFrame(dict(lk=META.loc[keys,"lesion_key"].values, y=META.loc[keys,"y"].values,
                                a=META.loc[keys,"a"].values, pid=META.loc[keys,"patient_id"].values, p=p))
            g=t.groupby("lk",sort=True)
            return (g.p.mean().values, g.y.max().values, g.a.first().values, g.pid.first().values)
        ptr,ytr,atr,_   = agg(ktr,str_);  btr = agg(ktr,Ptr[BASE])[0]
        pte,yte,ate,pat = agg(kte,ste_);  bte = agg(kte,Pte[BASE])[0]

    gt_b = fit_global(ytr,btr); gt_s = fit_global(ytr,ptr)
    tb,g0b = fit_bir(ytr,btr,atr); ts,g0s = fit_bir(ytr,ptr,atr)
    rows=[("OLD  two-stream + one threshold", bte,(bte>=gt_b).astype(int)),
          ("OLD  two-stream + Novelty 2",     bte, ap(bte,ate,tb,g0b)),
          ("NEW  selected score + one threshold", pte,(pte>=gt_s).astype(int)),
          ("NEW  selected score + Novelty 2  <== FINAL", pte, ap(pte,ate,ts,g0s))]
    print("\n"+"="*80)
    print("%s LEVEL — OFFICIAL TEST, n=%d (%.1f%% malignant)" % (level,len(yte),100*yte.mean()))
    print("="*80)
    print("  %-44s %-18s %-16s %-6s %-6s %s" % ("system","AUC [95% CI]","acc [95% CI]","sens","spec","missed"))
    for nm,sc,yh in rows:
        m=M(yte,sc,yh); la,ha=ci(yte,sc,yh,pat,"auc"); lc,hc=ci(yte,sc,yh,pat,"acc")
        print("  %-44s %.4f[%.3f-%.3f] %.1f%%[%.1f-%.1f] %.3f  %.3f  %d"
              % (nm,m["auc"],la,ha,100*m["acc"],100*lc,100*hc,m["sens"],m["spec"],m["fn"]))
        FINAL["%s | %s"%(level,nm)] = m
    print("\n  thresholds: %s (global %.2f)" % ({int(k):round(v,2) for k,v in sorted(ts.items())}, gt_s))
    print("\n  SPECIFICITY AT MATCHED SENSITIVITY  (refit on TRAIN at each target)")
    print("    %-8s %-24s %-24s %s" % ("target","one threshold","per-BI-RADS","gain"))
    for s in MATCH_SENS:
        g_s=fit_global(ytr,ptr,s); t_s,g0_=fit_bir(ytr,ptr,atr,s)
        m1=M(yte,pte,(pte>=g_s).astype(int)); m2=M(yte,pte,ap(pte,ate,t_s,g0_))
        print("    %-8.2f sens %.3f spec %.3f     sens %.3f spec %.3f     %+.3f"
              % (s,m1["sens"],m1["spec"],m2["sens"],m2["spec"],m2["spec"]-m1["spec"]))

with open(os.path.join(D,"FINAL_v2.json"),"w") as f:
    json.dump({"chosen":list(SEL),"how":HOW,"train_auc":SAUC,"results":FINAL}, f, indent=2, default=float)
print("\nsaved FINAL_v2.json")

PART A — AVAILABLE MEMBERS (train scores are out-of-fold)
   endtoend_fused                     n_tr 1318 n_te 378   TRAIN AUC 0.8398   test 0.8192
   endtoend                           n_tr 1318 n_te 378   TRAIN AUC 0.8442   test 0.8194
   handcrafted                        n_tr 1318 n_te 378   TRAIN AUC 0.6973   test 0.6781
   imageonly                          n_tr 1318 n_te 378   TRAIN AUC 0.7900   test 0.7994
   twostream_calcpre                  n_tr 1318 n_te 378   TRAIN AUC 0.8789   test 0.8582
   twostream_calcpre_cvmasks          n_tr 1318 n_te 378   TRAIN AUC 0.8811   test 0.8660
   twostream                          n_tr 1318 n_te 378   TRAIN AUC 0.8974   test 0.8769
   twostream_1seed                    n_tr 1318 n_te 378   TRAIN AUC 0.8974   test 0.8769

PART B — SEED CHECK
  officialsplit_oof vs _1seed : IDENTICAL -> current file is SINGLE seed, averaging is still available
  corr 1.00000 | mean abs diff 0.00000
  CV seeds: s11 AUC 0.8553 | s22 AUC 0.8713 | average 0.873

**`IL2` cell 1** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 5 — FINAL. OFFICIAL SPLIT, SENSITIVITY FLOOR 0.90
#
#   PROTOCOL (locked): fit everything on official TRAIN (1,318 img / 782 les),
#   freeze, report once on official TEST (378 img / 223 les). 0 patient overlap.
#   Train scores : cv_mass_twostream_officialtrain_oof.csv  (out-of-fold INSIDE
#                  official train — no official-test patient ever seen)
#   Test scores  : cv_mass_twostream_officialsplit_oof.csv
#
#   S1 CNN + one threshold      S2 CNN + per-BI-RADS   [NOVELTY 2]
#   S3 fusion + one threshold   S4 fusion + per-BI-RADS
#   All four honour the SAME sensitivity floor, so the comparison is fair.
#
#   No GPU, no training. ~3 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, json, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

D        = "/root/autodl-tmp/CBIS"
TRF      = "cv_mass_twostream_officialtrain_oof.csv"
TEF      = "cv_mass_twostream_officialsplit_oof.csv"
FLOOR    = 0.90                                   # <-- the clinical specification
SWEEP    = [0.70,0.75,0.80,0.85,0.88,0.90,0.92,0.95]
GRID     = np.round(np.arange(0.01,0.995,0.005),3)
MIN_N, MIN_POS, PASSES = 20, 3, 15
NBOOT, RNG = 2000, np.random.default_rng(7)

stem  = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
LOGIT = lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))

# ─── PART 1 : data ────────────────────────────────────────────────────────
print("="*80); print("PART 1 — INPUTS"); print("="*80)
d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
assert d["_k"].is_unique
for c in ("density","subtlety"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d["label"].astype(int)
d["a"]=pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["ds"]=pd.to_numeric(d["density"],errors="coerce");  d["ds"]=d["ds"].fillna(d["ds"].median())
d["sb"]=pd.to_numeric(d["subtlety"],errors="coerce"); d["sb"]=d["sb"].fillna(d["sb"].median())

def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); pc="prob" if "prob" in m.columns else \
      [c for c in m.columns if m[c].dtype.kind=="f" and m[c].between(0,1).all()][0]
    m["_k"]=m["img"].map(stem); return m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")

TR = d.merge(load(TRF),on="_k").reset_index(drop=True)
TE = d.merge(load(TEF),on="_k").reset_index(drop=True)
assert not (set(TR.patient_id) & set(TE.patient_id)), "patient overlap"
print("  TRAIN %4d img  %3d les  %3d pat  %.1f%% malig   (%s)"
      % (len(TR),TR.lesion_key.nunique(),TR.patient_id.nunique(),100*TR.y.mean(),TRF))
print("  TEST  %4d img  %3d les  %3d pat  %.1f%% malig   (%s)"
      % (len(TE),TE.lesion_key.nunique(),TE.patient_id.nunique(),100*TE.y.mean(),TEF))
print("  patient overlap 0  OK   |  sensitivity floor %.2f" % FLOOR)

def lesion(t):
    g=t.groupby("lesion_key",sort=True)
    return g.agg(p=("p","mean"),y=("y","max"),a=("a","first"),ds=("ds","first"),
                 sb=("sb","first"),patient_id=("patient_id","first")).reset_index()

# ─── PART 2 : machinery ───────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)

def fit_global(y,p,floor):
    """single threshold: best accuracy among cuts that meet the floor"""
    P,N=int(y.sum()),len(y)
    tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N, tp/max(P,1); ok=se>=floor-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])

def _asc(y,p,a,init,floor):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1) < floor-1e-12: return tau,-1.0,el          # infeasible start
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=floor-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j] > (TP+TN)/N + 1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N,el

def fit_bir(y,p,a,floor):
    """multi-start coordinate ascent; g0 start is feasible by construction"""
    g0=fit_global(y,p,floor); cats=[int(c) for c in np.unique(a)]
    fixed=[g0,0.05,0.15,0.25,0.35,0.45,0.50,0.55,0.65,max(0.01,g0-0.10),min(0.99,g0+0.10)]
    st=[{g:v for g in cats} for v in fixed]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba,bel=None,-np.inf,[]
    for s in st:
        t,ac,el=_asc(y,p,a,s,floor)
        if ac>ba+1e-12: best,ba,bel=t,ac,el
    if best is None or ba<0: best={g:g0 for g in cats}          # fall back to global
    return best,g0,bel,ba

ap = lambda p,a,t,g0: (p>=np.array([t.get(int(g),g0) for g in a])).astype(int)

def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),sens=tp/max(tp+fn,1),
                spec=tn/max(tn+fp,1),fp=fp,fn=fn)

def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))

CATS=[0,1,2,3,4,5]
def fuse(tr,te):
    X=lambda t: np.column_stack([LOGIT(t.p.values)]+
        [(t.a.values==g).astype(float) for g in CATS]+[t.ds.values,t.sb.values])
    Xtr,Xte=X(tr),X(te); mu,sd=Xtr.mean(0),Xtr.std(0)+1e-9
    lr=LogisticRegression(C=0.05,max_iter=5000).fit((Xtr-mu)/sd,tr.y.values)
    return lr.predict_proba((Xtr-mu)/sd)[:,1], lr.predict_proba((Xte-mu)/sd)[:,1]

# ─── PART 3 : primary result at FLOOR ─────────────────────────────────────
ROWS, FINAL = [], {}
for level,ftr,fte in (("IMAGE",TR,TE), ("LESION",lesion(TR),lesion(TE))):
    ytr,ptr,atr = ftr.y.values.astype(int), ftr.p.values, ftr.a.values.astype(int)
    yte,pte,ate = fte.y.values.astype(int), fte.p.values, fte.a.values.astype(int)
    pat = fte.patient_id.values
    ztr,zte = fuse(ftr,fte)

    print("\n"+"="*80)
    print("%s LEVEL — OFFICIAL TEST, n=%d (%.1f%% malignant, %d patients) | FLOOR %.2f"
          % (level,len(yte),100*yte.mean(),len(np.unique(pat)),FLOOR))
    print("  thresholds and fusion fitted on official TRAIN n=%d, then frozen" % len(ytr))
    print("="*80)

    g1 = fit_global(ytr,ptr,FLOOR)
    t2,g2,el2,tr2 = fit_bir(ytr,ptr,atr,FLOOR)
    g3 = fit_global(ytr,ztr,FLOOR)
    t4,g4,el4,_   = fit_bir(ytr,ztr,atr,FLOOR)
    sysd=[("S1  CNN + one threshold",              pte,(pte>=g1).astype(int)),
          ("S2  CNN + per-BI-RADS   [NOVELTY 2]",  pte, ap(pte,ate,t2,g2)),
          ("S3  fusion + one threshold",           zte,(zte>=g3).astype(int)),
          ("S4  fusion + per-BI-RADS",             zte, ap(zte,ate,t4,g4))]
    print("  %-40s %-18s %-16s %-6s %-6s %s"
          % ("system","AUC [95% CI]","acc [95% CI]","sens","spec","missed"))
    got={}
    for nm,sc,yh in sysd:
        m=M(yte,sc,yh); la,ha=ci(yte,sc,yh,pat,"auc"); lc,hc=ci(yte,sc,yh,pat,"acc")
        got[nm[:2]]=m; FINAL["%s|%s"%(level,nm[:2])]=dict(m,auc_ci=[la,ha],acc_ci=[lc,hc])
        star="  <== FINAL" if nm.startswith("S2") else ""
        print("  %-40s %.4f[%.3f-%.3f] %.1f%%[%.1f-%.1f] %.3f  %.3f  %d%s"
              % (nm,m["auc"],la,ha,100*m["acc"],100*lc,100*hc,m["sens"],m["spec"],m["fn"],star))

    print("\n  frozen thresholds  %s"
          % {int(k):round(v,3) for k,v in sorted(t2.items())})
    print("  categories with their own threshold: %s   (others use global %.3f)"
          % (el2,g2))
    print("  train accuracy of the fitted layer: %.4f" % tr2)
    print("\n  S2 vs S1  (your layer vs one cut, same score, same floor) : %+.2f acc pts, %+d cancers found"
          % (100*(got["S2"]["acc"]-got["S1"]["acc"]), got["S1"]["fn"]-got["S2"]["fn"]))
    print("  S2 vs S3  (your layer vs radiologist-assessment fusion)    : %+.2f acc pts, %+d cancers found"
          % (100*(got["S2"]["acc"]-got["S3"]["acc"]), got["S3"]["fn"]-got["S2"]["fn"]))
    print("  S4 vs S3  (redundancy: layer on top of BI-RADS fusion)     : %+.2f acc pts"
          % (100*(got["S4"]["acc"]-got["S3"]["acc"])))

    # ─── PART 4 : floor sweep ─────────────────────────────────────────────
    print("\n  FLOOR SWEEP — everything refitted on TRAIN at each floor")
    print("    %-6s %-24s %-24s %s" % ("floor","S1 one threshold","S2 per-BI-RADS","gain"))
    for fl in SWEEP:
        ga=fit_global(ytr,ptr,fl); tb,gb,_,_=fit_bir(ytr,ptr,atr,fl)
        m1=M(yte,pte,(pte>=ga).astype(int)); m2=M(yte,pte,ap(pte,ate,tb,gb))
        ROWS.append(dict(level=level,floor=fl,
                         s1_acc=m1["acc"],s1_sens=m1["sens"],s1_spec=m1["spec"],s1_fn=m1["fn"],
                         s2_acc=m2["acc"],s2_sens=m2["sens"],s2_spec=m2["spec"],s2_fn=m2["fn"]))
        print("    %-6.2f acc %5.1f%% se %.3f sp %.3f   acc %5.1f%% se %.3f sp %.3f   %+.2f pts, %+d cancers"
              % (fl,100*m1["acc"],m1["sens"],m1["spec"],
                 100*m2["acc"],m2["sens"],m2["spec"],
                 100*(m2["acc"]-m1["acc"]), m1["fn"]-m2["fn"]))

pd.DataFrame(ROWS).to_csv(os.path.join(D,"floor_sweep_official.csv"),index=False)
with open(os.path.join(D,"FINAL_floor090.json"),"w") as f:
    json.dump(dict(protocol="official TCIA split; fit on train, frozen; report on test",
                   floor=FLOOR,train_scores=TRF,test_scores=TEF,results=FINAL),
              f,indent=2,default=float)
print("\nsaved floor_sweep_official.csv  and  FINAL_floor090.json")

PART 1 — INPUTS
  TRAIN 1318 img  782 les  691 pat  48.3% malig   (cv_mass_twostream_officialtrain_oof.csv)
  TEST   378 img  223 les  201 pat  38.9% malig   (cv_mass_twostream_officialsplit_oof.csv)
  patient overlap 0  OK   |  sensitivity floor 0.90

IMAGE LEVEL — OFFICIAL TEST, n=378 (38.9% malignant, 201 patients) | FLOOR 0.90
  thresholds and fusion fitted on official TRAIN n=1318, then frozen
  system                                   AUC [95% CI]       acc [95% CI]     sens   spec   missed
  S1  CNN + one threshold                  0.8769[0.825-0.919] 75.7%[70.7-80.5] 0.884  0.675  17
  S2  CNN + per-BI-RADS   [NOVELTY 2]      0.8769[0.828-0.918] 81.0%[76.3-85.4] 0.878  0.766  18  <== FINAL
  S3  fusion + one threshold               0.9061[0.861-0.942] 79.1%[73.9-84.1] 0.884  0.732  17
  S4  fusion + per-BI-RADS                 0.9061[0.866-0.943] 80.2%[75.1-84.9] 0.884  0.749  17

  frozen thresholds  {0: 0.515, 1: 0.44, 2: 0.44, 3: 0.495, 4: 0.455, 5: 0.01}
  categories with t

**`IL2` cell 2** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 6 — COMPLETE OPERATING GRID, OFFICIAL TEST SET   (fixed)
#
#   4 systems  x  8 sensitivity floors  x  4 reporting units
#   Everything fitted on official TRAIN and frozen. Test touched for scoring only.
#
#   units : IMAGE 378 | LESION 223 | BREAST ~210 | PATIENT 201
#   S1 CNN + one threshold     S2 CNN + per-BI-RADS  [NOVELTY 2]
#   S3 fusion + one threshold  S4 fusion + per-BI-RADS
#
#   No GPU, no training. ~4 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, json, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

D     = "/root/autodl-tmp/CBIS"
TRF   = "cv_mass_twostream_officialtrain_oof.csv"
TEF   = "cv_mass_twostream_officialsplit_oof.csv"
SWEEP = [0.70,0.75,0.80,0.85,0.88,0.90,0.92,0.95]
GRID  = np.round(np.arange(0.01,0.995,0.005),3)
MIN_N, MIN_POS, PASSES = 20, 3, 15
NBOOT, RNG = 2000, np.random.default_rng(7)

stem  = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
LOGIT = lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))

# ─── data ─────────────────────────────────────────────────────────────────
d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
assert d["_k"].is_unique
for c in ("density","subtlety","side"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d["label"].astype(int)
d["a"]=pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["ds"]=pd.to_numeric(d["density"],errors="coerce");  d["ds"]=d["ds"].fillna(d["ds"].median())
d["sb"]=pd.to_numeric(d["subtlety"],errors="coerce"); d["sb"]=d["sb"].fillna(d["sb"].median())
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)

def load(fn):
    m=pd.read_csv(os.path.join(D,fn))
    pc="prob" if "prob" in m.columns else [c for c in m.columns
        if m[c].dtype.kind=="f" and m[c].between(0,1).all()][0]
    m["_k"]=m["img"].map(stem); return m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")

TR=d.merge(load(TRF),on="_k").reset_index(drop=True)
TE=d.merge(load(TEF),on="_k").reset_index(drop=True)
assert not (set(TR.patient_id)&set(TE.patient_id))

def roll(t,key):
    """aggregate to a reporting unit; never re-aggregate the grouping column"""
    if key is None: return t.reset_index(drop=True)
    spec = dict(p=("p","mean"), y=("y","max"), a=("a","max"),
                ds=("ds","first"), sb=("sb","first"))
    if key != "patient_id":
        spec["patient_id"] = ("patient_id","first")
    out = t.groupby(key, sort=True).agg(**spec).reset_index()
    if "patient_id" not in out.columns:
        out["patient_id"] = out[key].values
    return out

UNITS=[("IMAGE",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")]

# ─── machinery ────────────────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TP+TN)/N+1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,0.05,0.15,0.25,0.35,0.45,0.50,0.55,0.65,
                                      max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),sens=tp/max(tp+fn,1),
                spec=tn/max(tn+fp,1),fn=fn)
def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))
CATS=[0,1,2,3,4,5]
def fuse(tr,te):
    X=lambda t: np.column_stack([LOGIT(t.p.values)]+
        [(t.a.values==g).astype(float) for g in CATS]+[t.ds.values,t.sb.values])
    Xtr,Xte=X(tr),X(te); mu,sd=Xtr.mean(0),Xtr.std(0)+1e-9
    lr=LogisticRegression(C=0.05,max_iter=5000).fit((Xtr-mu)/sd,tr.y.values)
    return lr.predict_proba((Xtr-mu)/sd)[:,1],lr.predict_proba((Xte-mu)/sd)[:,1]

# ─── the grid ─────────────────────────────────────────────────────────────
ALL=[]
for uname,key in UNITS:
    ftr,fte=roll(TR,key),roll(TE,key)
    ytr,ptr,atr=ftr.y.values.astype(int),ftr.p.values,ftr.a.values.astype(int)
    yte,pte,ate=fte.y.values.astype(int),fte.p.values,fte.a.values.astype(int)
    ztr,zte=fuse(ftr,fte)
    print("\n"+"="*92)
    print("%s LEVEL — OFFICIAL TEST n=%d (%.1f%% malignant) | train n=%d | AUC cnn %.4f  fusion %.4f"
          % (uname,len(yte),100*yte.mean(),len(ytr),
             roc_auc_score(yte,pte),roc_auc_score(yte,zte)))
    print("="*92)
    print("  %-6s %-22s %-22s %-22s %-22s"
          % ("floor","S1 cnn+one","S2 cnn+BI-RADS [YOURS]","S3 fus+one","S4 fus+BI-RADS"))
    for fl in SWEEP:
        g1=fit_global(ytr,ptr,fl); t2,g2=fit_bir(ytr,ptr,atr,fl)
        g3=fit_global(ytr,ztr,fl); t4,g4=fit_bir(ytr,ztr,atr,fl)
        out=[]
        for tag,sc,yh in (("S1",pte,(pte>=g1).astype(int)),
                          ("S2",pte,ap(pte,ate,t2,g2)),
                          ("S3",zte,(zte>=g3).astype(int)),
                          ("S4",zte,ap(zte,ate,t4,g4))):
            m=M(yte,sc,yh); out.append(m)
            ALL.append(dict(unit=uname,n=len(yte),floor=fl,system=tag,
                            **{k:m[k] for k in ("auc","acc","sens","spec","fn")}))
        print("  %-6.2f %s" % (fl, " ".join(
            "%5.1f%% se%.3f sp%.3f " % (100*m["acc"],m["sens"],m["spec"]) for m in out)))

R=pd.DataFrame(ALL); R.to_csv(os.path.join(D,"operating_grid_official.csv"),index=False)

# ─── best configurations, with intervals ──────────────────────────────────
print("\n"+"="*92); print("HIGHEST-ACCURACY CONFIGURATION PER UNIT, SYSTEM S2 (your Novelty 2)"); print("="*92)
for uname,key in UNITS:
    sub=R[(R.unit==uname)&(R.system=="S2")].sort_values("acc",ascending=False).iloc[0]
    s1 =R[(R.unit==uname)&(R.system=="S1")&(R.floor==sub.floor)].iloc[0]
    ftr,fte=roll(TR,key),roll(TE,key)
    ytr,ptr,atr=ftr.y.values.astype(int),ftr.p.values,ftr.a.values.astype(int)
    yte,pte,ate=fte.y.values.astype(int),fte.p.values,fte.a.values.astype(int)
    t2,g2=fit_bir(ytr,ptr,atr,sub.floor); yh=ap(pte,ate,t2,g2)
    la,ha=ci(yte,pte,yh,fte.patient_id.values,"auc")
    lc,hc=ci(yte,pte,yh,fte.patient_id.values,"acc")
    print("  %-8s n=%-4d floor %.2f   AUC %.4f[%.3f-%.3f]  acc %.1f%%[%.1f-%.1f]  se %.3f sp %.3f  missed %d   (vs one threshold %.1f%%, %+.2f pts)"
          % (uname,int(sub.n),sub.floor,sub.auc,la,ha,100*sub.acc,100*lc,100*hc,
             sub.sens,sub.spec,int(sub.fn),100*s1.acc,100*(sub.acc-s1.acc)))
print("\nsaved operating_grid_official.csv")


IMAGE LEVEL — OFFICIAL TEST n=378 (38.9% malignant) | train n=1318 | AUC cnn 0.8769  fusion 0.9061
  floor  S1 cnn+one             S2 cnn+BI-RADS [YOURS] S3 fus+one             S4 fus+BI-RADS        
  0.70    81.7% se0.755 sp0.857   82.0% se0.830 sp0.814   84.1% se0.796 sp0.870   82.5% se0.755 sp0.870 
  0.75    81.7% se0.755 sp0.857   83.1% se0.782 sp0.861   84.1% se0.796 sp0.870   82.5% se0.755 sp0.870 
  0.80    80.2% se0.823 sp0.788   81.7% se0.837 sp0.805   84.1% se0.796 sp0.870   82.5% se0.755 sp0.870 
  0.85    79.4% se0.823 sp0.775   81.7% se0.837 sp0.805   81.2% se0.871 sp0.775   82.5% se0.816 sp0.831 
  0.88    77.0% se0.857 sp0.714   82.0% se0.871 sp0.788   80.2% se0.878 sp0.753   81.2% se0.871 sp0.775 
  0.90    75.7% se0.884 sp0.675   81.0% se0.878 sp0.766   79.1% se0.884 sp0.732   80.2% se0.884 sp0.749 
  0.92    73.3% se0.905 sp0.623   78.3% se0.905 sp0.706   77.8% se0.905 sp0.697   77.8% se0.905 sp0.697 
  0.95    65.3% se0.959 sp0.459   73.5% se0.952 sp0.597   73.3% 

**`IL2` cell 3** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [6]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 7 — AGGREGATION RULE + NOVELTY-1 LADDER, ALL FOUR LEVELS
#
#   PROTOCOL (unchanged): official TCIA split. Everything fitted on official
#   TRAIN (691 patients) and frozen. Official TEST (201 patients) scored once.
#
#   PART 1  three models: plain DenseNet -> mask-pooling -> two-stream
#   PART 2  how to combine several lesions into a breast / a patient
#           (7 rules, chosen on TRAIN AUC only)
#   PART 3  Novelty 1 ladder at every level, with the chosen rule
#   PART 4  final result: chosen rule + Novelty 2, floor 0.90, with 95% CIs
#
#   No GPU, no training. ~4 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

D      = "/root/autodl-tmp/CBIS"
FLOOR  = 0.90
GRID   = np.round(np.arange(0.01,0.995,0.005),3)
MIN_N, MIN_POS, PASSES = 20, 3, 15
NBOOT, RNG = 2000, np.random.default_rng(7)

MODELS = [
 ("baseline  plain DenseNet-121", "cv_mass_imageonly_officialtrain_oof.csv",
                                  "cv_mass_imageonly_officialsplit_oof.csv"),
 ("N1a       + mask-weighted pooling", "cv_mass_v2_officialtrain_oof.csv",
                                       "cv_mass_officialsplit_oof.csv"),
 ("N1b       + two-stream  [FINAL]", "cv_mass_twostream_officialtrain_oof.csv",
                                     "cv_mass_twostream_officialsplit_oof.csv"),
]
FINALM = "N1b       + two-stream  [FINAL]"

stem  = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
LOGIT = lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))
SIG   = lambda z: 1/(1+np.exp(-z))

# ─── PART 1 : data + models ───────────────────────────────────────────────
print("="*88); print("PART 1 — MODELS"); print("="*88)
d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
assert d["_k"].is_unique
if "side" not in d.columns:
    fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
    d=d.merge(fx[["_k","side"]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d["label"].astype(int)
d["a"]=pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
d["sp"]=np.where(d["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
BASE = d.set_index("_k")[["y","a","lesion_key","breast_key","patient_id","sp"]]

def load(fn):
    f=os.path.join(D,fn)
    if not os.path.exists(f): return None
    m=pd.read_csv(f)
    if "img" not in m.columns: return None
    pc="prob" if "prob" in m.columns else None
    if pc is None:
        c=[x for x in m.columns if m[x].dtype.kind=="f" and m[x].between(0,1).all()]
        if not c: return None
        pc=c[0]
    m["_k"]=m["img"].map(stem)
    m=m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")
    return m[m["_k"].isin(BASE.index)].set_index("_k")["p"]

SC={}
for nm,trf,tef in MODELS:
    A,B = load(trf), load(tef)
    if A is None or B is None:
        print("   %-36s MISSING (%s / %s)" % (nm,trf,tef)); continue
    ya,yb = BASE.loc[A.index,"y"].values, BASE.loc[B.index,"y"].values
    SC[nm]=(A,B)
    print("   %-36s train %4d (AUC %.4f)   TEST %3d (AUC %.4f)"
          % (nm,len(A),roc_auc_score(ya,A.values),len(B),roc_auc_score(yb,B.values)))
assert FINALM in SC, "final model predictions not found"
print("\n   sanity: image-level TEST AUC should read ~0.799 / ~0.848 / 0.877 top to bottom.")
print("   if the middle row is far off, the N1a file pairing is wrong — tell me and I'll refit it.")

# ─── PART 2 : aggregation rules ───────────────────────────────────────────
RULES = ([("mean",None),("max",None),("min",None),("logit_mean",None),("noisy_or",None),
          ("top2",None)] + [("power",q) for q in (2,3,4,6)])

def combine(arr, rule, par):
    if len(arr)==1: return float(arr[0])
    if rule=="mean":       return float(arr.mean())
    if rule=="max":        return float(arr.max())
    if rule=="min":        return float(arr.min())
    if rule=="logit_mean": return float(SIG(LOGIT(arr).mean()))
    if rule=="noisy_or":   return float(1-np.prod(1-arr))
    if rule=="top2":       return float(np.sort(arr)[-2:].mean())
    if rule=="power":      return float(np.mean(np.clip(arr,1e-6,1)**par)**(1.0/par))
    raise ValueError(rule)

def build(series, key, rule, par):
    """collapse image scores to one row per key using `rule`"""
    t = pd.DataFrame(dict(p=series.values, k=BASE.loc[series.index,key].values,
                          y=BASE.loc[series.index,"y"].values,
                          a=BASE.loc[series.index,"a"].values,
                          pid=BASE.loc[series.index,"patient_id"].values))
    g = t.groupby("k",sort=True)
    out = g.agg(y=("y","max"), a=("a","max"), pid=("pid","first")).reset_index()
    arrs = {k: v.values for k,v in g["p"]}
    out["p"] = [combine(arrs[k], rule, par) for k in out.k]
    out["nimg"] = [len(arrs[k]) for k in out.k]
    return out

LEVELS = [("IMAGE",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")]
Atr, Ate = SC[FINALM]
Atr_tr = Atr[BASE.loc[Atr.index,"sp"].eq("train").values]
Ate_te = Ate[BASE.loc[Ate.index,"sp"].eq("test").values]

print("\n"+"="*88); print("PART 2 — HOW TO COMBINE SEVERAL IMAGES INTO ONE UNIT"); print("="*88)
CHOSEN={}
for lv,key in LEVELS:
    if key is None: CHOSEN[lv]=("mean",None); continue
    rows=[]
    for r,pa in RULES:
        tr = build(Atr_tr,key,r,pa); te = build(Ate_te,key,r,pa)
        rows.append((r,pa,roc_auc_score(tr.y,tr.p),roc_auc_score(te.y,te.p)))
    tr0 = build(Atr_tr,key,"mean",None); te0 = build(Ate_te,key,"mean",None)
    multi_tr = int((tr0.nimg>1).sum()); multi_te = int((te0.nimg>1).sum())
    print("\n  %s   test units %d (%d built from >1 image)   train units %d (%d multi)"
          % (lv,len(te0),multi_te,len(tr0),multi_tr))
    print("    %-14s %-6s %-12s %s" % ("rule","par","TRAIN AUC","test AUC"))
    for r,pa,at,ae in sorted(rows,key=lambda z:-z[2]):
        mk = "  <-- chosen" if (r,pa)==max(rows,key=lambda z:z[2])[:2] else ""
        print("    %-14s %-6s %.4f       %.4f%s" % (r,"" if pa is None else pa,at,ae,mk))
    best = max(rows,key=lambda z:z[2])
    CHOSEN[lv]=(best[0],best[1])
print("\n  chosen per level (on TRAIN AUC only): %s" % {k:v[0] for k,v in CHOSEN.items()})

# ─── PART 3 : Novelty 1 ladder ────────────────────────────────────────────
print("\n"+"="*88); print("PART 3 — NOVELTY 1 LADDER (test AUC, chosen aggregation)"); print("="*88)
print("  %-36s %-9s %-9s %-9s %-9s" % ("model","IMAGE","LESION","BREAST","PATIENT"))
LAD={}
for nm,_,_ in MODELS:
    if nm not in SC: continue
    _,B = SC[nm]; Bte = B[BASE.loc[B.index,"sp"].eq("test").values]
    cells=[]
    for lv,key in LEVELS:
        r,pa = CHOSEN[lv]
        t = Bte.to_frame("p") if key is None else None
        if key is None:
            au = roc_auc_score(BASE.loc[Bte.index,"y"].values, Bte.values)
        else:
            u = build(Bte,key,r,pa); au = roc_auc_score(u.y,u.p)
        cells.append(au); LAD[(nm,lv)]=au
    print("  %-36s %s" % (nm," ".join("%.4f   " % c for c in cells)))
print("\n  Novelty 1 total gain (baseline -> N1b)")
for lv,_ in LEVELS:
    b,f = LAD.get((MODELS[0][0],lv)), LAD.get((FINALM,lv))
    if b and f: print("    %-8s %+.4f AUC" % (lv, f-b))

# ─── PART 4 : final result with Novelty 2 ─────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TP+TN)/N+1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,0.05,0.15,0.25,0.35,0.45,0.50,0.55,0.65,
                                      max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),sens=tp/max(tp+fn,1),
                spec=tn/max(tn+fp,1),fp=fp,fn=fn)
def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))

print("\n"+"="*88)
print("PART 4 — FINAL RESULT, floor %.2f, chosen aggregation, OFFICIAL TEST" % FLOOR)
print("="*88)
print("  %-8s %-5s %-6s %-19s %-18s %-6s %-6s %-7s %s"
      % ("level","n","rule","AUC [95% CI]","acc [95% CI]","sens","spec","missed","vs 1 threshold"))
OUT={}
for lv,key in LEVELS:
    r,pa = CHOSEN[lv]
    if key is None:
        ytr,ptr,atr = (BASE.loc[Atr_tr.index,"y"].values, Atr_tr.values,
                       BASE.loc[Atr_tr.index,"a"].values)
        yte,pte,ate = (BASE.loc[Ate_te.index,"y"].values, Ate_te.values,
                       BASE.loc[Ate_te.index,"a"].values)
        pat = BASE.loc[Ate_te.index,"patient_id"].values
    else:
        u,v = build(Atr_tr,key,r,pa), build(Ate_te,key,r,pa)
        ytr,ptr,atr = u.y.values, u.p.values, u.a.values
        yte,pte,ate = v.y.values, v.p.values, v.a.values
        pat = v.pid.values
    g1 = fit_global(ytr,ptr,FLOOR); t2,g2 = fit_bir(ytr,ptr,atr,FLOOR)
    m1 = M(yte,pte,(pte>=g1).astype(int)); m2 = M(yte,pte,ap(pte,ate,t2,g2))
    la,ha = ci(yte,pte,ap(pte,ate,t2,g2),pat,"auc")
    lc,hc = ci(yte,pte,ap(pte,ate,t2,g2),pat,"acc")
    OUT[lv]=dict(rule=r,par=pa,**m2,auc_ci=[la,ha],acc_ci=[lc,hc],
                 acc_one_threshold=m1["acc"])
    print("  %-8s %-5d %-6s %.4f[%.3f-%.3f] %.1f%%[%.1f-%.1f] %.3f  %.3f  %2d/%-4d %.1f%% (%+.2f)"
          % (lv,len(yte),r,m2["auc"],la,ha,100*m2["acc"],100*lc,100*hc,
             m2["sens"],m2["spec"],m2["fn"],int(yte.sum()),
             100*m1["acc"],100*(m2["acc"]-m1["acc"])))
    print("           thresholds %s" % {int(k):round(v,3) for k,v in sorted(t2.items())})

with open(os.path.join(D,"FINAL_v3.json"),"w") as f:
    json.dump(dict(floor=FLOOR,aggregation={k:v[0] for k,v in CHOSEN.items()},
                   ladder={"%s|%s"%k:v for k,v in LAD.items()},results=OUT),
              f,indent=2,default=float)
print("\nsaved FINAL_v3.json")

PART 1 — MODELS
   baseline  plain DenseNet-121         train 1318 (AUC 0.7900)   TEST 378 (AUC 0.7994)
   N1a       + mask-weighted pooling    train 1318 (AUC 0.8675)   TEST 378 (AUC 0.8480)
   N1b       + two-stream  [FINAL]      train 1318 (AUC 0.8974)   TEST 378 (AUC 0.8769)

   sanity: image-level TEST AUC should read ~0.799 / ~0.848 / 0.877 top to bottom.
   if the middle row is far off, the N1a file pairing is wrong — tell me and I'll refit it.

PART 2 — HOW TO COMBINE SEVERAL IMAGES INTO ONE UNIT

  LESION   test units 223 (155 built from >1 image)   train units 782 (536 multi)
    rule           par    TRAIN AUC    test AUC
    power          4      0.9099       0.8977  <-- chosen
    power          3      0.9098       0.8999
    power          2      0.9094       0.9019
    power          6      0.9091       0.8949
    mean                  0.9081       0.9043
    top2                  0.9081       0.9043
    logit_mean            0.9081       0.9049
    max                  

**`SEG` cell 11** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL E v4 — FINAL RESULT: fit on official TRAIN, report on official TEST
#   Pool restricted to models consistent with the OFFICIAL masks:
#     v2, two-stream, two-stream calcpre  (official masks)
#     imageonly                            (uses no mask at all)
#   handcrafted / endtoend are excluded: their features came from the
#   old CV masks, so they are not part of the official-mask pipeline.
#   Nothing is fitted on the 223 reported lesions.  CPU, ~30 s.
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix

D          = "/root/autodl-tmp/CBIS"
SENS_FLOOR = 0.70
LEAK_AUC   = 0.97
HEADLINE   = "cv_mass_twostream"          # the single model reported as the main result

MEMBERS = ["cv_mass_v2", "cv_mass_twostream",
           "cv_mass_twostream_calcpre", "cv_mass_imageonly"]
TEST_FILE = {"cv_mass_v2": "cv_mass_officialsplit_oof.csv"}
test_f  = lambda n: TEST_FILE.get(n, "%s_officialsplit_oof.csv" % n)
train_f = lambda n: "%s_officialtrain_oof.csv" % n

stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
d["_k"] = d["img"].map(stem)
assert d["_k"].is_unique
K2L = dict(zip(d["_k"], d["lesion_key"]))
K2Y = dict(zip(d["_k"], d["label"].astype(int)))
K2A = dict(zip(d["_k"], pd.to_numeric(d["assessment"], errors="coerce")))
K2S = dict(zip(d["_k"], d["official_split"].astype(str).str.lower()))

def load(fn, want_test):
    f = os.path.join(D, fn)
    if not os.path.exists(f): return None
    m = pd.read_csv(f); m["_k"] = m["img"].map(stem)
    m = m[m["_k"].isin(K2L)].copy()
    if not len(m): return None
    m["lesion_key"] = m["_k"].map(K2L); m["y"] = m["_k"].map(K2Y)
    m["a"] = m["_k"].map(K2A);          m["sp"] = m["_k"].map(K2S)
    m = m[m["sp"].str.contains("test") == want_test]
    if not len(m): return None
    g = m.groupby("lesion_key").agg(p=("prob","mean"), y=("y","max"),
                                    a=("a","first")).reset_index()
    g["a"] = pd.to_numeric(g.a, errors="coerce").fillna(4).astype(int).clip(0,5)
    return g

names, TE, TR = [], None, None
for nm in MEMBERS:
    t, r = load(test_f(nm), True), load(train_f(nm), False)
    if t is None: print("  absent (no test file) : %s" % nm); continue
    if r is None: print("  EXCLUDED (no train file): %s" % nm); continue
    assert roc_auc_score(t.y, t.p) < LEAK_AUC, "leakage guard on %s" % nm
    names.append(nm)
    TE = t[["lesion_key","y","a"]].copy() if TE is None else TE
    TR = r[["lesion_key","y","a"]].copy() if TR is None else TR
    TE = TE.merge(t[["lesion_key","p"]].rename(columns={"p":nm}), on="lesion_key")
    TR = TR.merge(r[["lesion_key","p"]].rename(columns={"p":nm}), on="lesion_key")
assert names, "no member has both a train-side and a test-side file"
assert HEADLINE in names, "%s is missing — cannot report the single-model headline" % HEADLINE

yte, ate = TE.y.values.astype(int), TE.a.values
ytr, atr = TR.y.values.astype(int), TR.a.values
assert not (set(TE.lesion_key) & set(TR.lesion_key)), "a lesion is in both sets"

print("\n%-30s %-11s %s" % ("member", "TRAIN AUC", "TEST AUC"))
print("-" * 56)
for nm in names:
    print("%-30s %-11.4f %.4f"
          % (nm, roc_auc_score(ytr, TR[nm].values), roc_auc_score(yte, TE[nm].values)))
print("\nTRAIN %d lesions (%.0f%% malignant) | TEST %d lesions (%.0f%% malignant)"
      % (len(ytr), 100*ytr.mean(), len(yte), 100*yte.mean()))

GRID = np.round(np.arange(0.02, 0.99, 0.01), 3)
def acc_of(y, yh): return (yh == y).mean()
def fit_tau(y, p, a, floor, passes=12):
    tau = {g: 0.50 for g in np.unique(a)}
    ap = lambda t: (p >= np.array([t.get(g, 0.50) for g in a])).astype(int)
    b = acc_of(y, ap(tau))
    for _ in range(passes):
        moved = False
        for g in tau:
            for c in GRID:
                t2 = dict(tau); t2[g] = c; yh = ap(t2)
                if yh[y == 1].mean() < floor: continue
                s = acc_of(y, yh)
                if s > b + 1e-9: tau, b, moved = t2, s, True
        if not moved: break
    return tau
def show(name, yh):
    tn, fp, fn, tp = confusion_matrix(yte, yh, labels=[0,1]).ravel()
    pr, rc = tp/max(tp+fp,1), tp/max(tp+fn,1)
    f1, ac = 2*pr*rc/max(pr+rc,1e-9), (tp+tn)/len(yte)
    print("  %-42s acc %.1f%%  sens %.3f  spec %.3f  F1 %.3f  FP %2d  missed %2d"
          % (name, 100*ac, rc, tn/max(tn+fp,1), f1, fp, fn))
    return ac, f1, fn

def report(title, ptr, pte):
    print("\n" + "=" * 86); print(title); print("=" * 86)
    auc = roc_auc_score(yte, pte)
    gt  = GRID[int(np.argmax([acc_of(ytr, (ptr >= t).astype(int)) for t in GRID]))]
    show("single global threshold (%.2f, from TRAIN)" % gt, (pte >= gt).astype(int))
    tau = fit_tau(ytr, ptr, atr, SENS_FLOOR)
    ac, f1, fn = show("per-BI-RADS  [NOVELTY 2]  (from TRAIN)",
                      (pte >= np.array([tau.get(g, 0.50) for g in ate])).astype(int))
    print("\n  AUC %.4f | accuracy %.1f%% | F1 %.3f | missed %d" % (auc, 100*ac, f1, fn))
    print("  thresholds (from TRAIN): %s" % {int(k): float(v) for k, v in sorted(tau.items())})
    return auc, ac, f1, fn

# ── A. the single model — this is the headline ────────────────────────
a1 = report("A.  SINGLE MODEL — two-stream (official masks)",
            TR[HEADLINE].values, TE[HEADLINE].values)

# ── B. greedy ensemble — the upper bound ──────────────────────────────
mixT = lambda s: np.column_stack([TR[m].values for m in s]).mean(1)
mixE = lambda s: np.column_stack([TE[m].values for m in s]).mean(1)
chosen, best = [], -np.inf
print("\ngreedy forward selection on official TRAIN:")
while True:
    pk, pa = None, best
    for nm in names:
        if nm in chosen: continue
        a = roc_auc_score(ytr, mixT(chosen + [nm]))
        if a > pa + 1e-6: pk, pa = nm, a
    if pk is None: break
    chosen.append(pk); best = pa
    print("   + %-30s train AUC -> %.4f" % (pk, best))
a2 = report("B.  ENSEMBLE — %s" % " + ".join(chosen), mixT(chosen), mixE(chosen))

print("\n" + "=" * 86); print("SUMMARY"); print("=" * 86)
print("  %-34s %-8s %-9s %-8s %s" % ("row", "AUC", "accuracy", "F1", "missed"))
print("  %-34s %-8.4f %-9.1f %-8.3f %d" % ("single model (two-stream)", a1[0], 100*a1[1], a1[2], a1[3]))
print("  %-34s %-8.4f %-9.1f %-8.3f %d" % ("ensemble (%d members)" % len(chosen), a2[0], 100*a2[1], a2[2], a2[3]))
print("  %-34s %-8.4f %-9.1f %-8s %s" % ("your five-fold CV", 0.9089, 87.0, "-", "-"))
print("\n  Nothing above was fitted on the 223 reported lesions.")


member                         TRAIN AUC   TEST AUC
--------------------------------------------------------
cv_mass_v2                     0.8860      0.8715
cv_mass_twostream              0.9081      0.9043
cv_mass_twostream_calcpre      0.8933      0.8867
cv_mass_imageonly              0.8045      0.8092

TRAIN 782 lesions (48% malignant) | TEST 223 lesions (39% malignant)

A.  SINGLE MODEL — two-stream (official masks)
  single global threshold (0.50, from TRAIN) acc 85.2%  sens 0.816  spec 0.875  F1 0.811  FP 17  missed 16
  per-BI-RADS  [NOVELTY 2]  (from TRAIN)     acc 86.5%  sens 0.816  spec 0.897  F1 0.826  FP 14  missed 16

  AUC 0.9043 | accuracy 86.5% | F1 0.826 | missed 16
  thresholds (from TRAIN): {0: 0.52, 1: 0.5, 2: 0.85, 3: 0.58, 4: 0.5, 5: 0.02}

greedy forward selection on official TRAIN:
   + cv_mass_twostream              train AUC -> 0.9081
   + cv_mass_twostream_calcpre      train AUC -> 0.9115
   + cv_mass_v2                     train AUC -> 0.9134

B.  ENSEMB

**`SEG` cell 14** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [11]:
# ══════════════════════════════════════════════════════════════════════
# CONFIDENCE INTERVALS + ABLATION SIGNIFICANCE  (CPU, ~1 min)
#   DeLong 95% CI on AUC, DeLong paired tests between models,
#   bootstrap 95% CIs on accuracy / F1 / sensitivity / specificity
#   at the thresholds fitted on the official TRAINING partition.
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, confusion_matrix

D, SENS_FLOOR, NBOOT, SEED = "/root/autodl-tmp/CBIS", 0.70, 2000, 42
MODELS = {"two-stream":         "cv_mass_twostream",
          "v2 (single)":        "cv_mass_v2",
          "image-only":         "cv_mass_imageonly",
          "two-stream calcpre": "cv_mass_twostream_calcpre"}
TEST_FILE = {"cv_mass_v2": "cv_mass_officialsplit_oof.csv"}
tf = lambda n: TEST_FILE.get(n, "%s_officialsplit_oof.csv" % n)
rf = lambda n: "%s_officialtrain_oof.csv" % n

stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
d["_k"] = d["img"].map(stem)
K = {"lesion_key": dict(zip(d["_k"], d["lesion_key"])),
     "y":  dict(zip(d["_k"], d["label"].astype(int))),
     "a":  dict(zip(d["_k"], pd.to_numeric(d["assessment"], errors="coerce"))),
     "sp": dict(zip(d["_k"], d["official_split"].astype(str).str.lower()))}

def load(fn, want_test):
    f = os.path.join(D, fn)
    if not os.path.exists(f): return None
    m = pd.read_csv(f); m["_k"] = m["img"].map(stem)
    m = m[m["_k"].isin(K["lesion_key"])].copy()
    for c in K: m[c] = m["_k"].map(K[c])
    m = m[m["sp"].str.contains("test") == want_test]
    if not len(m): return None
    g = m.groupby("lesion_key").agg(p=("prob","mean"), y=("y","max"),
                                    a=("a","first")).reset_index()
    g["a"] = pd.to_numeric(g.a, errors="coerce").fillna(4).astype(int).clip(0,5)
    return g

TE, TR, have = None, None, []
for label, nm in MODELS.items():
    t, r = load(tf(nm), True), load(rf(nm), False)
    if t is None:
        print("  missing test file : %s" % label); continue
    have.append(label)
    TE = t[["lesion_key","y","a"]].copy() if TE is None else TE
    TE = TE.merge(t[["lesion_key","p"]].rename(columns={"p":label}), on="lesion_key")
    if r is not None:
        TR = r[["lesion_key","y","a"]].copy() if TR is None else TR
        TR = TR.merge(r[["lesion_key","p"]].rename(columns={"p":label}), on="lesion_key")
Y, A = TE.y.values.astype(int), TE.a.values
print("official TEST: %d lesions, %d malignant (%.0f%%)\n" % (len(Y), Y.sum(), 100*Y.mean()))

# ---------------- DeLong ----------------
def midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2

def _structural(y, plist):
    order = np.argsort(-y)
    y2 = y[order]; m = int(y2.sum()); n = len(y2) - m
    P = np.vstack([p[order] for p in plist])
    tx = np.vstack([midrank(P[r, :m]) for r in range(P.shape[0])])
    ty = np.vstack([midrank(P[r, m:]) for r in range(P.shape[0])])
    tz = np.vstack([midrank(P[r, :])  for r in range(P.shape[0])])
    aucs = tz[:, :m].sum(1)/m/n - (m + 1.0)/2/n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    return aucs, v01, v10, m, n

def delong_ci(y, p, alpha=0.05):
    a, v01, v10, m, n = _structural(y, [p])
    var = v01[0].var(ddof=1)/m + v10[0].var(ddof=1)/n
    z = norm.ppf(1 - alpha/2); se = np.sqrt(var)
    return a[0], max(0.0, a[0] - z*se), min(1.0, a[0] + z*se)

def delong_test(y, p1, p2):
    a, v01, v10, m, n = _structural(y, [p1, p2])
    S = np.cov(v01)/m + np.cov(v10)/n
    diff = a[0] - a[1]
    var = S[0,0] + S[1,1] - 2*S[0,1]
    if var <= 0: return diff, float("nan")
    z = diff/np.sqrt(var)
    return diff, 2*(1 - norm.cdf(abs(z)))

print("%-22s %-8s DeLong 95%% CI" % ("model", "AUC"))
print("-"*52)
for label in have:
    a, lo, hi = delong_ci(Y, TE[label].values)
    print("%-22s %-8.4f [%.4f, %.4f]" % (label, a, lo, hi))

print("\nablation comparisons (DeLong paired test on the same %d lesions)" % len(Y))
print("-"*72)
for a_, b_, what in [("image-only", "v2 (single)", "mask-weighted pooling"),
                     ("v2 (single)", "two-stream", "wide-context branch"),
                     ("image-only", "two-stream", "full pipeline vs unguided")]:
    if a_ in have and b_ in have:
        diff, p = delong_test(Y, TE[b_].values, TE[a_].values)
        ptxt = "n/a" if p != p else ("%.2e" % p if p < 1e-3 else "%.4f" % p)
        print("  %-28s %+.4f   p = %s" % (what, diff, ptxt))

# ---------------- bootstrap CIs at the TRAIN-fitted thresholds ----------------
HEAD = "two-stream"
assert HEAD in have and TR is not None and HEAD in TR.columns, "need the two-stream train file"
GRID = np.round(np.arange(0.02, 0.99, 0.01), 3)
ytr, atr, ptr = TR.y.values.astype(int), TR.a.values, TR[HEAD].values

def fit(y, p, a, floor, passes=12):
    tau = {g: 0.50 for g in np.unique(a)}
    ap = lambda t: (p >= np.array([t.get(g, 0.50) for g in a])).astype(int)
    b = (ap(tau) == y).mean()
    for _ in range(passes):
        moved = False
        for g in tau:
            for c in GRID:
                t2 = dict(tau); t2[g] = c; yh = ap(t2)
                if yh[y == 1].mean() < floor: continue
                s = (yh == y).mean()
                if s > b + 1e-9: tau, b, moved = t2, s, True
        if not moved: break
    return tau

tau = fit(ytr, ptr, atr, SENS_FLOOR)
P  = TE[HEAD].values
YH = (P >= np.array([tau.get(g, 0.50) for g in A])).astype(int)

def metrics(y, yh):
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    pr, rc = tp/max(tp+fp,1), tp/max(tp+fn,1)
    return dict(accuracy=(tp+tn)/len(y), sensitivity=rc,
                specificity=tn/max(tn+fp,1), F1=2*pr*rc/max(pr+rc,1e-9),
                FP=fp, missed=fn)
obs = metrics(Y, YH)

rng = np.random.default_rng(SEED)
boot = {k: [] for k in ["accuracy","sensitivity","specificity","F1","FP","missed","AUC"]}
N = len(Y)
for _ in range(NBOOT):
    idx = rng.integers(0, N, N)
    if len(set(Y[idx])) < 2: continue
    mm = metrics(Y[idx], YH[idx])
    for k in ["accuracy","sensitivity","specificity","F1","FP","missed"]:
        boot[k].append(mm[k])
    boot["AUC"].append(roc_auc_score(Y[idx], P[idx]))

print("\n%s at thresholds fitted on the official TRAINING partition" % HEAD)
print("%-14s %-10s bootstrap 95%% CI (%d resamples)" % ("metric", "value", NBOOT))
print("-"*60)
print("%-14s %-10.4f [%.4f, %.4f]"
      % ("AUC", roc_auc_score(Y, P), *np.percentile(boot["AUC"], [2.5, 97.5])))
for k in ["accuracy","sensitivity","specificity","F1"]:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    print("%-14s %-10.4f [%.4f, %.4f]" % (k, obs[k], lo, hi))
for k in ["FP","missed"]:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    print("%-14s %-10d [%d, %d]" % (k, obs[k], int(round(lo)), int(round(hi))))
print("\n  thresholds used: %s" % {int(a_): float(b_) for a_, b_ in sorted(tau.items())})
print("  DeLong and bootstrap AUC intervals should agree closely.")

official TEST: 223 lesions, 87 malignant (39%)

model                  AUC      DeLong 95% CI
----------------------------------------------------
two-stream             0.9043   [0.8626, 0.9461]
v2 (single)            0.8715   [0.8231, 0.9198]
image-only             0.8092   [0.7505, 0.8680]
two-stream calcpre     0.8867   [0.8417, 0.9318]

ablation comparisons (DeLong paired test on the same 223 lesions)
------------------------------------------------------------------------
  mask-weighted pooling        +0.0622   p = 6.95e-04
  wide-context branch          +0.0329   p = 0.0082
  full pipeline vs unguided    +0.0951   p = 1.75e-05

two-stream at thresholds fitted on the official TRAINING partition
metric         value      bootstrap 95% CI (2000 resamples)
------------------------------------------------------------
AUC            0.9043     [0.8594, 0.9431]
accuracy       0.8655     [0.8161, 0.9103]
sensitivity    0.8161     [0.7294, 0.8904]
specificity    0.8971     [0.8433, 0.94

## E · Official-split evaluation from the earlier pipeline


**`WPN` cell 48** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 33 — EVALUATE ON THE OFFICIAL CBIS-DDSM TRAIN/TEST SPLIT
#   Thresholds are fitted ONLY on official-train lesions and applied
#   unchanged to official-test lesions. CPU only, ~1 min.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D, LES = "/root/autodl-tmp/CBIS", "mass"
GRID = np.round(np.arange(0.02, 0.981, 0.01), 3)

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES))
d["lesion_key"] = d["lesion_key"].astype(str)
assert "official_split" in d.columns, "no official_split column"
d["osp"] = d["official_split"].astype(str).str.lower().str.strip()
print("official_split values:", d["osp"].value_counts().to_dict())

sp = d.groupby("lesion_key")["osp"].agg(lambda s: s.mode().iat[0])
L = pd.read_csv(os.path.join(D, "mass_lesion_errors_final.csv"))
L["lesion_key"] = L["lesion_key"].astype(str)
L["osp"] = L["lesion_key"].map(sp)
L = L.dropna(subset=["osp"]).reset_index(drop=True)

istr = L["osp"].str.contains("train")
iste = L["osp"].str.contains("test")
print("\nlesions: official-train %d | official-test %d | unmapped %d"
      % (istr.sum(), iste.sum(), len(L) - istr.sum() - iste.sum()))
ov = set(L.loc[istr, "pid"]) & set(L.loc[iste, "pid"])
print("patients in BOTH official train and test: %d  %s" % (len(ov), "(must be 0)" if not ov else "<-- PROBLEM"))

y, p, ass = L["y"].values, L["p"].values, L["ass"].values

def sens(yy, pr): return ((pr==1)&(yy==1)).sum()/max((yy==1).sum(),1)
def curves(yy, pp, gg):
    out = {}
    for g in np.unique(gg):
        m = gg==g; ys, ps = yy[m], pp[m]
        pr = ps[None,:] > GRID[:,None]
        out[g] = (np.asarray((pr & (ys==1)).sum(1)), np.asarray((~pr & (ys==0)).sum(1)))
    return out
def fit(cur, P, floor, passes=8):
    gs = list(cur.keys())
    tp_a = np.sum([cur[g][0] for g in gs], 0); cr_a = np.sum([cur[g][0]+cur[g][1] for g in gs], 0)
    ok = (tp_a/max(P,1)) >= floor
    s0 = int(np.argmax(np.where(ok, cr_a, -1))) if ok.any() else int(np.argmax(cr_a))
    idx = {g: s0 for g in gs}
    ttp = int(sum(cur[g][0][idx[g]] for g in gs))
    tcr = int(sum(cur[g][0][idx[g]]+cur[g][1][idx[g]] for g in gs))
    for _ in range(passes):
        moved = False
        for g in gs:
            tp, tn = cur[g]
            n_tp = ttp - tp[idx[g]] + tp
            n_cr = tcr - (tp[idx[g]]+tn[idx[g]]) + tp + tn
            fe = (n_tp/max(P,1)) >= floor
            if not fe.any(): continue
            j = int(np.argmax(np.where(fe, n_cr, -1)))
            if j != idx[g]: idx[g]=j; ttp=int(n_tp[j]); tcr=int(n_cr[j]); moved=True
        if not moved: break
    return idx

FLOOR = 0.60
idx = fit(curves(y[istr.values], p[istr.values], ass[istr.values]),
          int((y[istr.values]==1).sum()), FLOOR)
dflt = int(np.median(list(idx.values())))
print("\nthresholds fitted on official-train:",
      {int(g): float(GRID[i]) for g, i in sorted(idx.items())})

def report(mask, tag):
    yy, pp, gg = y[mask], p[mask], ass[mask]
    pr = np.zeros(len(yy), int)
    for g in np.unique(gg):
        m = gg==g; pr[m] = (pp[m] > GRID[idx.get(g, dflt)]).astype(int)
    tn, fp, fn, tp = confusion_matrix(yy, pr, labels=[0,1]).ravel()
    print("  %-26s n=%4d  malig %.0f%%  AUC %.4f  acc %.1f%%  sens %.3f  spec %.3f  FP %3d  FN %3d"
          % (tag, len(yy), 100*yy.mean(), roc_auc_score(yy, pp), 100*accuracy_score(yy, pr),
             tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))
    return pr

print("\n" + "="*104)
print("OFFICIAL CBIS-DDSM SPLIT")
print("="*104)
report(istr.values, "official TRAIN (in-sample)")
prt = report(iste.values, "official TEST (held out)")
report(np.ones(len(L), bool), "all lesions (5-fold CV)")

print("\n--- official TEST, by BI-RADS ---")
yt, at = y[iste.values], ass[iste.values]
rows = []
for g in np.unique(at):
    m = at==g
    if m.sum() < 5: continue
    a_,b_,c_,d_ = confusion_matrix(yt[m], prt[m], labels=[0,1]).ravel()
    rows.append(dict(BIRADS=int(g), n=int(m.sum()), malig=int((yt[m]==1).sum()),
                     AUC=round(roc_auc_score(yt[m], p[iste.values][m]),3) if len(set(yt[m]))>1 else np.nan,
                     acc="%.1f%%" % (100*accuracy_score(yt[m], prt[m])), FP=b_, missed=c_))
print(pd.DataFrame(rows).to_string(index=False))

print("\nIMPORTANT — how to describe this in the thesis:")
print("  These are out-of-fold predictions from patient-grouped 5-fold CV, RESTRICTED to the")
print("  official test lesions, with thresholds fitted only on the official training lesions.")
print("  Every lesion is still scored by a model that never saw its patient, so the number is")
print("  leakage-free. It is NOT a reproduction of the official protocol (which would require")
print("  retraining on the official training set alone) — call it 'official test subset under")
print("  our CV protocol', not 'official split result'.")

official_split values: {'train': 1318, 'test': 378}

lesions: official-train 782 | official-test 223 | unmapped 0
patients in BOTH official train and test: 0  (must be 0)

thresholds fitted on official-train: {0: 0.54, 1: 0.02, 2: 0.8, 3: 0.59, 4: 0.48, 5: 0.02}

OFFICIAL CBIS-DDSM SPLIT
  official TRAIN (in-sample) n= 782  malig 48%  AUC 0.9150  acc 87.6%  sens 0.888  spec 0.865  FP  55  FN  42
  official TEST (held out)   n= 223  malig 39%  AUC 0.8791  acc 81.6%  sens 0.816  spec 0.816  FP  25  FN  16
  all lesions (5-fold CV)    n=1005  malig 46%  AUC 0.9077  acc 86.3%  sens 0.874  spec 0.853  FP  80  FN  58

--- official TEST, by BI-RADS ---
 BIRADS  n  malig   AUC   acc  FP  missed
      0 18      2 0.906 77.8%   3       1
      2 10      1 1.000 90.0%   0       1
      3 53      2 0.980 92.5%   4       0
      4 96     38 0.750 68.8%  16      14
      5 45     43 0.640 95.6%   2       0

IMPORTANT — how to describe this in the thesis:
  These are out-of-fold predictions from pati

## F · Mask provenance check


**`IL1` cell 1** — import os, numpy as np, pandas as pd  
<sub>1 output block(s) preserved</sub>


In [1]:
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
if "side" not in d.columns:
    fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
    d=d.merge(fx[["_k","side"]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int); d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
B=d.set_index("_k")

for fn in ("cv_mass_twostream_officialsplit_oof.csv",
           "cv_mass_twostream_officialsplit_oof_cvmasks.csv"):
    f=os.path.join(D,fn)
    if not os.path.exists(f): print("MISSING %s"%fn); continue
    m=pd.read_csv(f); m["_k"]=m["img"].map(stem); m=m[m["_k"].isin(B.index)]
    row=[fn.replace("cv_mass_twostream_officialsplit_oof","").replace(".csv","") or "  (standard)"]
    for nm,key in (("ROI",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")):
        t=pd.DataFrame(dict(p=m.prob.values,y=B.loc[m._k,"y"].values,
                            k=(m._k.values if key is None else B.loc[m._k,key].values)))
        g=t.groupby("k").agg(p=("p","mean"),y=("y","max"))
        row.append("%s %.4f"%(nm,roc_auc_score(g.y,g.p)))
    print("  ".join(row))
print("\nIf the two lines are close, mask provenance is not a problem.")
print("If _cvmasks is clearly lower, the standard file used masks from a segmentation")
print("model that saw the official-test patients, and _cvmasks is what you must report.")

  (standard)  ROI 0.8769  LESION 0.9043  BREAST 0.9016  PATIENT 0.9041
_cvmasks  ROI 0.8695  LESION 0.9004  BREAST 0.8948  PATIENT 0.8967

If the two lines are close, mask provenance is not a problem.
If _cvmasks is clearly lower, the standard file used masks from a segmentation
model that saw the official-test patients, and _cvmasks is what you must report.
